# Code With SUZH

### Intermediate Python

**From Comfortable Programmer to Python Developer**

**By SUZH TEAM**
**Presented By Muhammad Usman Rouf, CEO**

---

This picks up exactly where the Basics volume left off. You already know how to move data around, branch, loop, write functions, catch exceptions, and read and write files. None of that gets retaught here — where an idea in this book leans on something from Basics, it'll get a short reminder rather than a full re-explanation.

What changes in this volume is the *kind* of question worth asking. Basics was mostly about "what syntax produces what result." Intermediate is about a smaller number of deeper questions — what is an object, really; what does Python do when a `for` loop runs; what does `@decorator` actually rewrite — asked precisely enough that the answers generalize far past the specific example in front of you.

Python 3.11+ throughout, same as before.

# Part I — Writing More Expressive Python

## 31. Comprehensions — Thinking in Expressions

Here's an entirely ordinary piece of code, the kind you'd have written constantly in Basics: 

In [ ]:
numbers = [1, 2, 3, 4, 5]

squares = []
for n in numbers:
    squares.append(n ** 2)

print(squares)

Nothing wrong with it. But look at what it's actually saying, stripped of mechanics: *build a new list, containing `n ** 2`, for each `n` in `numbers`.* Three lines of loop machinery — creating an empty list, looping, appending — exist entirely to serve that one idea. A **list comprehension** lets you write the idea directly: 

In [ ]:
squares = [n ** 2 for n in numbers]
print(squares)

Read it left to right as a sentence, not as a mysterious inversion of the loop: "give me `n ** 2`, for each `n` in `numbers`." The part before `for` is the expression that gets computed each time; the part from `for` onward is exactly the same iteration you'd write in a normal loop.

### The shape underneath it

Every list comprehension has the same three parts, and it's worth naming them once so the rest of the chapter can refer back to them:

```text
[ expression   for   target   in   iterable ]
     ↓                  ↓            ↓
 what to build     name for each   what to loop over
```

This isn't new syntax bolted onto the language — it's the `for` loop you already know, restructured so the expression comes first. Given that, the translation between the two forms is completely mechanical, and it's worth being able to do it in both directions in your head before trusting a comprehension you didn't write yourself.

### Adding a condition

A loop that only keeps some values usually looks like this: 

In [ ]:
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

evens = []
for n in numbers:
    if n % 2 == 0:
        evens.append(n)

print(evens)

A comprehension expresses the same filter with a trailing `if`: 

In [ ]:
evens = [n for n in numbers if n % 2 == 0]
print(evens)

Notice the `if` here comes *after* the `for` and has no `else` — it's a filter, deciding which values get included at all, not a value-producing expression. That's a genuinely different `if` from the one you're about to see next.

### An `if` that chooses a value, instead of filtering

Sometimes you want to transform every value differently depending on a condition, keeping all of them, rather than dropping some. That needs a conditional *expression*, not a filter, and it goes in a different position — before the `for`, as part of what gets computed.

In [ ]:
labels = ["even" if n % 2 == 0 else "odd" for n in numbers]
print(labels)

`"even" if n % 2 == 0 else "odd"` is itself a complete expression — a conditional expression, sometimes called a ternary — that evaluates to one string or the other. Comprehensions don't add anything new here; they simply accept any Python expression in the first slot, and this happens to be one. It's worth keeping these two `if`s mentally distinct: an `if` after the `for` filters which items survive; an `if`/`else` before the `for`, inside the expression itself, picks between two computed values without dropping anything.

### Nested comprehensions

You can loop over more than one thing, in the same order you'd nest the equivalent `for` loops: 

In [ ]:
pairs = []
for x in range(1, 3):
    for y in range(1, 3):
        pairs.append((x, y))

print(pairs)

In [ ]:
pairs = [(x, y) for x in range(1, 3) for y in range(1, 3)]
print(pairs)

Read the clauses left to right in the same order they'd appear as nested loops — the leftmost `for` is the outer loop, exactly as it was above. Beyond two levels, this stops being more readable than the loop it replaces, and it's worth switching back to an ordinary nested loop the moment a comprehension needs a mental diagram to parse. A comprehension that requires you to trace it slowly has already lost its main advantage, which is being readable at a glance.

### Set and dictionary comprehensions

The same idea extends to the other two collection types you already know, using `{}` instead of `[]`, distinguished by whether there's a `:` inside.

In [ ]:
words = ["apple", "banana", "cherry", "date"]

unique_lengths = {len(word) for word in words}
print(unique_lengths)

In [ ]:
lengths_by_word = {word: len(word) for word in words}
print(lengths_by_word)

`{len(word) for word in words}` — no colon — is a **set comprehension**, producing a set of values (with duplicates automatically collapsed, exactly as any set would). `{word: len(word) for word in words}` — with a colon — is a **dictionary comprehension**, producing key-value pairs. Same underlying mechanism as list comprehensions throughout; only the container and, for dictionaries, the extra `key: value` shape at the front change.

### When a normal loop is actually the better choice

A comprehension is an expression that must produce a single collection. The moment your loop body needs to do more than one thing per item — print something *and* accumulate a result, update two different structures, break out early, or handle an exception — a comprehension is fighting the tool rather than using it well.

In [ ]:
# Awkward: comprehension abused for its side effects, throwing away the actual list it builds
[print(n) for n in range(3)]

# Clear: a loop, used for what loops are for
for n in range(3):
    print(n)

The first version works, but the list it builds (`[None, None, None]`, since `print` returns `None`) is pure waste — nobody wants that list, it exists only because a comprehension is required to produce one. If you find yourself ignoring a comprehension's result, that's the sign you didn't actually want a comprehension. The rule of thumb worth keeping: reach for a comprehension when the goal is genuinely *build a new collection from an existing iterable*, and reach for an ordinary loop the moment the goal is anything else.

### One problem

> Given `words = ["hi", "hello", "hey", "greetings", "yo"]`, use a single dictionary comprehension to produce a dictionary mapping each word to `True` if its length is greater than 3, and `False` otherwise.

In [ ]:
words = ["hi", "hello", "hey", "greetings", "yo"]

# TODO: build the dictionary described above using one comprehension


## 32. Unpacking and Extended Unpacking

Basics briefly showed this:

In [ ]:
point = (3, 4)
x, y = point
print(x, y)

That's **unpacking** — distributing a sequence's values into separate names in one step, rather than indexing them out one at a time (`x = point[0]`, `y = point[1]`). This chapter goes further than Basics did, because unpacking turns out to have more shapes than just "same number of names as values."

### The problem unpacking solves

Without it, pulling several related values out of a sequence is tedious and easy to get subtly wrong: 

In [ ]:
record = ("Ali", 25, "Lahore")

name = record[0]
age = record[1]
city = record[2]
print(name, age, city)

In [ ]:
name, age, city = record
print(name, age, city)

Same result, but the second version states directly what's happening — three names, matched positionally against three values — rather than making you verify three separate index numbers are each correct.

### `*` — collecting the rest

Basic unpacking requires the number of names to match the number of values exactly, or Python raises an error. `*` relaxes that, by letting one name absorb everything that doesn't fit the other names.

In [ ]:
numbers = [1, 2, 3, 4, 5]

first, *rest = numbers
print(first)
print(rest)

In [ ]:
first, *middle, last = numbers
print(first)
print(middle)
print(last)

`*rest` and `*middle` always come back as **lists**, regardless of what kind of sequence you unpacked from — that's worth noting explicitly, since the input here was already a list; try it with a tuple and the starred name is still a list. Only one starred name is allowed per unpacking; Python has no way to know how to split "the rest" between two greedy names, so it doesn't let you ask.

### `*` in a function call — the reverse direction

Unpacking isn't only for assignment. `*` in front of a sequence, used as a function argument, spreads that sequence out into separate positional arguments: 

In [ ]:
def add_three(a, b, c):
    return a + b + c

values = [1, 2, 3]
print(add_three(*values))

Without the `*`, `add_three(values)` would pass the *entire list* as a single argument `a`, and fail — `add_three` expects three separate arguments, not one list. `*values` unpacks the list at the call site, so Python sees `add_three(1, 2, 3)` instead. This is the same underlying idea as unpacking into names — spreading a sequence into separate slots — just applied to a function call's argument list rather than an assignment's left-hand side.

### `**` — the same idea for dictionaries

`**` performs the equivalent spreading operation for dictionaries, matching keys to parameter names: 

In [ ]:
def describe(name, age):
    print(f"{name} is {age} years old")

info = {"name": "Sara", "age": 30}
describe(**info)

`**info` spreads the dictionary's entries out as keyword arguments — equivalent to writing `describe(name="Sara", age=30)` directly — matching each key to the parameter of the same name. The keys in the dictionary have to match the function's actual parameter names exactly, or Python has nothing to match them against and raises an error.

### Nested unpacking

Because unpacking is really just pattern-matching a shape, that shape can be nested, mirroring the structure of what's being unpacked: 

In [ ]:
record = ("Ali", (25, "Lahore"))

name, (age, city) = record
print(name, age, city)

The parentheses around `(age, city)` on the left mirror the parentheses around `(25, "Lahore")` on the right — Python matches the *shape* of the left-hand side against the *shape* of the value being unpacked, one level at a time, however deep that shape goes.

### One problem

> Given `scores = (95, 88, 76, 60, 100, 82)`, unpack it so that `highest` holds the first score, `lowest` holds the last score, and `middle_scores` holds everything in between, using a single unpacking statement with `*`.

In [ ]:
scores = (95, 88, 76, 60, 100, 82)

# TODO: unpack into highest, middle_scores, lowest using * -- careful about the order


## 33. Iterables — The Idea Behind `for`

You've written dozens of `for` loops by now, over strings, lists, ranges, dictionaries. It's worked every time, on every one of those types, without you ever needing to ask *how*. This chapter asks exactly that question, because the answer is the foundation for a large part of what's ahead — generators, in particular, will make very little sense without it.

### What actually happens when `for` runs

Take an ordinary loop: 

In [ ]:
for letter in "cat":
    print(letter)

`for` didn't inspect `"cat"` and special-case "oh, this is a string, I know how those work." Instead, `for` performs a small, fixed protocol against *whatever* it's given, and that protocol happens to be something strings support. Here's that protocol, written out manually, using two functions you've never had a reason to call directly before: 

In [ ]:
text = "cat"

iterator = iter(text)
print(iterator)

print(next(iterator))
print(next(iterator))
print(next(iterator))

`iter(text)` produces something new — an **iterator** — which is a distinct object from the string itself, specifically responsible for remembering "how far through have we gotten." Each call to `next()` asks that iterator for the next value and moves it one step forward. Call `next()` a fourth time, past the end: 

In [ ]:
print(next(iterator))

`StopIteration` — a signal, not a bug, that there's nothing left. And this is the entire mechanism a `for` loop is built on:

```text
for
 ↓
iter(object)
 ↓
iterator
 ↓
next(iterator)
 ↓
value
 ↓
repeat, until StopIteration
```

A `for` loop calls `iter()` once, at the start, to get an iterator; then calls `next()` on that iterator repeatedly, using each returned value as the loop variable, until `next()` raises `StopIteration` — at which point the loop catches that signal quietly and stops, without it ever looking like an error to you. You've been watching this happen every time you've written `for`, without the machinery ever being visible.

### Iterable versus iterator — a distinction worth keeping precise

These two words get used almost interchangeably in casual conversation, and it's worth being exact about the difference, because generators (two chapters from now) live entirely on the iterator side of it.

An **iterable** is anything `iter()` can be called on successfully — anything that knows how to *produce* an iterator. A string is iterable. A list is iterable. A dictionary is iterable.

An **iterator** is the object `iter()` hands back — the thing that actually remembers position and responds to `next()`.

The distinction matters because of what you can do with each: an iterable can be iterated over *repeatedly*, from the start, every time — each `for` loop over the same list calls `iter()` fresh and gets a new iterator starting at the beginning. An iterator, once partially consumed, stays consumed — there's no way to rewind it.

In [ ]:
numbers = [1, 2, 3]

for n in numbers:
    print(n)

# The list itself is unaffected -- iterating it again starts fresh.
for n in numbers:
    print(n)

In [ ]:
numbers = [1, 2, 3]
it = iter(numbers)

print(next(it))
print(next(it))

# The iterator remembers where it left off -- it does NOT restart.
for n in it:
    print(n)

That last loop only printed `3` — the iterator `it` had already given up `1` and `2` via the two explicit `next()` calls, and a `for` loop over an already-partially-consumed iterator picks up exactly where it left off, because a `for` loop's first step, calling `iter()`, is actually a no-op here: `iter()` called on an iterator that's already an iterator just returns that same iterator, unchanged, rather than starting a new one. This is precisely why an iterator can only be consumed once from start to finish, while a genuinely iterable *collection* — a list, a string — can be iterated over again and again, because each attempt calls `iter()` on the *collection*, not on some already-in-progress iterator.

### Why this matters beyond satisfying curiosity

Every collection type you already know — string, list, tuple, dictionary, set — is iterable specifically because each one implements this same small protocol (something that responds correctly to `iter()`). `for` never needed separate logic for each type; it needed exactly one protocol, and every iterable type agrees to support it. That's the actual payoff of understanding the mechanism: once you know the shape of the protocol, you already know most of what's needed to build your own iterable type — which is precisely where Chapter 44 picks this back up, once classes are available to build one.

### One problem

> Using only `iter()` and `next()` (no `for` loop), print every character of the string `"loop"`, then show that a fifth call to `next()` raises `StopIteration` — catch that with `try`/`except` and print `"done"` instead of letting it crash.

In [ ]:
word = "loop"

# TODO: manually iterate through word using iter() and next(),
# catching StopIteration on the call past the end


# Part II — Python's Object Model

## 34. Classes and Objects — Building Your Own Types

Suppose you're tracking books in a small library program. A dictionary handles one book reasonably: 

In [ ]:
book = {"title": "Dune", "author": "Frank Herbert", "pages": 412, "read": False}
print(book["title"], "by", book["author"])

Now write a function that marks a book as read: 

In [ ]:
def mark_as_read(book):
    book["read"] = True

mark_as_read(book)
print(book["read"])

This works, but notice what's *not* enforced anywhere: nothing stops a second dictionary from being built with a typo — `"titel"` instead of `"title"` — and Python won't warn you until something tries to read the missing key and fails, possibly far from where the dictionary was created. Nothing groups "the shape a book is supposed to have" and "the operations you can do to a book" into one place; the dictionary's structure lives as an unstated convention, and `mark_as_read` lives somewhere else entirely, connected to it only by the fact that a programmer remembered to write it that way.

A **class** is Python's tool for fixing exactly this: it lets you define a *type* — a description of what data a thing has, and what it can do — and then create as many actual objects of that type as you need, each guaranteed to have the same shape.

In [ ]:
class Book:
    def __init__(self, title, author, pages):
        self.title = title
        self.author = author
        self.pages = pages
        self.read = False

    def mark_as_read(self):
        self.read = True

Nothing printed — same as `def` on its own, this only *defines* the class; it doesn't create anything yet. Creating an actual book looks like a function call: 

In [ ]:
dune = Book("Dune", "Frank Herbert", 412)

print(dune.title)
print(dune.author)
print(dune.read)

dune.mark_as_read()
print(dune.read)

### Vocabulary worth being precise about

`Book` is the **class** — the type, the blueprint. `dune` is an **object** (equivalently, an **instance** of `Book`) — one actual thing built from that blueprint, with its own data. `title`, `author`, `pages`, and `read` are **attributes** — named pieces of data attached to a specific object. `mark_as_read` is a **method** — a function attached to the class, meant to be called on a specific object.

You've actually used all of this vocabulary before without the formal names: every list you've ever created is an *object*, an instance of the `list` *class*; `.append()` is a *method*; and `list` itself was a class the whole time, just one Python provided for you rather than one you wrote. Building your own class means you're no longer limited to the shapes Python ships with.

### Why this is better than the dictionary version

Two books, side by side, each with independent data: 

In [ ]:
dune = Book("Dune", "Frank Herbert", 412)
hobbit = Book("The Hobbit", "J.R.R. Tolkien", 310)

dune.mark_as_read()

print(dune.title, dune.read)
print(hobbit.title, hobbit.read)

Every `Book` object is guaranteed to have `title`, `author`, `pages`, and `read` — there's no way to accidentally create one with a typo'd attribute name and not notice, the way there was with a dictionary key, because `__init__` (next chapter's subject) runs the same setup code every time, for every object. And `mark_as_read` lives on the class itself, right next to the data it operates on, rather than as a loose function elsewhere that merely *happens* to expect a dictionary shaped a certain way.

### Instance namespace versus class namespace, briefly

Each object you create — `dune`, `hobbit` — has its own private storage for its attributes (`title`, `author`, and so on), separate from every other object's storage. But `mark_as_read` isn't duplicated for each book; it's defined once, on the class, and shared by every instance. This split — data that's per-object, behavior that's shared on the class — is worth noticing now as a shape, because the next three chapters are essentially a much closer look at exactly this split.

### One problem

> Write a class `Rectangle` with `width` and `height` set in `__init__`, and a method `area()` that returns the rectangle's area. Create two rectangles and print both of their areas.

In [ ]:
# TODO: define Rectangle, create two instances, print their areas


## 35. `self`, `__init__`, and Instance State

The `Book` class from the last chapter had `self` appear in every method, and `__init__` in particular looked a little strange the first time — a method whose name is surrounded by double underscores, called automatically, never written explicitly at the call site. This chapter takes both apart properly.

### What `__init__` actually is

`__init__` is not special syntax — it's an ordinary method, with an unusual name that Python treats as significant: when you call a class as though it were a function (`Book("Dune", "Frank Herbert", 412)`), Python creates a new, essentially empty object first, and then calls that object's `__init__` automatically, passing along whatever arguments you gave. `__init__`'s job is to take that freshly created, blank object and set it up — attach whatever starting attributes it needs.

```text
Book("Dune", "Frank Herbert", 412)
        ↓
a new, empty Book object is created
        ↓
__init__ is called automatically on that new object
        ↓
__init__'s body runs, attaching attributes to it
        ↓
the finished object is handed back to you
```

This is worth separating explicitly because the words are easy to conflate: **construction** (making the object exist at all) happens first and is mostly automatic; **initialization** (`__init__`, setting the object up) happens second and is the part you actually write.

### What `self` represents

Here's the same method, called two different ways, to make `self` concrete rather than abstract: 

In [ ]:
class Book:
    def __init__(self, title):
        self.title = title

    def shout_title(self):
        print(self.title.upper() + "!")

dune = Book("Dune")
hobbit = Book("The Hobbit")

dune.shout_title()
hobbit.shout_title()

`dune.shout_title()` and `hobbit.shout_title()` call the exact same method — there's only one `shout_title` function, defined once on the class — yet each call operates on a different book's title. `self` is how: when you call `dune.shout_title()`, Python actually calls `Book.shout_title(dune)` behind the scenes, passing `dune` in as the first argument. `self` inside the method's body is simply that argument's name — a name you chose (by convention, everyone chooses `self`), referring to "whichever specific object this call was made on."

You can watch this directly by calling the method the unbound way, through the class rather than an instance, and seeing that it requires exactly the argument `self` normally receives automatically: 

In [ ]:
Book.shout_title(dune)   # identical to dune.shout_title()

`dune.shout_title()` is really just more convenient syntax for `Book.shout_title(dune)` — the dot-call automatically supplies the object on the left of the dot as the method's first argument. That's the entire mechanism; there's no additional magic beyond "the object you called the method on gets silently passed in as the first parameter," and `self` is simply the conventional name for catching it.

### Why `self` has to be written explicitly

Some languages hide this parameter entirely. Python's design deliberately keeps it visible, on the reasoning that "which object is this operating on" is important enough information that a reader shouldn't have to remember an invisible rule to know it's there — every method's parameter list tells you, honestly, exactly what it receives, `self` included.

### Instance attributes are genuinely separate per object

Because `self` refers to a specific object, `self.title = title` inside `__init__` attaches `title` to *that particular object's* own storage, not to some shared location.

In [ ]:
dune = Book("Dune")
hobbit = Book("The Hobbit")

dune.title = "Dune (Extended Edition)"

print(dune.title)
print(hobbit.title)

Changing `dune.title` had zero effect on `hobbit.title` — each instance's attributes live in that instance's own private namespace, exactly as promised at the end of the last chapter. This is the object-oriented version of the aliasing-versus-independence distinction from Basics: two separate `Book` objects are two separate things, the same way two separately created lists are two separate lists, even if they were built from identical starting data.

### One problem

> Write a class `Counter` with `__init__` setting a `count` attribute to `0`, and methods `increment()` (adds 1) and `reset()` (sets back to 0). Create one counter, increment it three times, print its count, reset it, and print again.

In [ ]:
# TODO: define Counter and exercise it as described


## 36. Class Attributes vs Instance Attributes

Every attribute in the last two chapters was set inside `__init__`, using `self.something = ...` — an **instance attribute**, private to that one object. There's a second kind, attached directly to the class itself.

In [ ]:
class Book:
    library_name = "SUZH Community Library"

    def __init__(self, title):
        self.title = title

dune = Book("Dune")
hobbit = Book("The Hobbit")

print(dune.library_name)
print(hobbit.library_name)

`library_name` was defined once, directly in the class body, outside of any method — it isn't attached to any particular object, it's attached to the *class* `Book` itself. Yet both `dune.library_name` and `hobbit.library_name` found it. That's **attribute lookup**: when you write `dune.library_name`, Python first checks whether `dune` itself has an attribute by that name; if not, it checks the class `dune` was built from.

```text
dune.library_name
      │
      ├── does dune's own instance namespace have "library_name"?  → no
      │
      └── does Book (dune's class) have "library_name"?  → yes, found it
```

Every `Book` instance shares the exact same `library_name` value, because there's only one — living on the class, not duplicated per instance.

### Shadowing

Here's what happens if an instance *does* set an attribute with the same name: 

In [ ]:
dune.library_name = "Ali's Personal Collection"

print(dune.library_name)
print(hobbit.library_name)
print(Book.library_name)

`dune.library_name = ...` created a *new* instance attribute on `dune` specifically, which now **shadows** the class attribute for `dune` alone — lookup on `dune` finds the instance attribute first and never even checks the class. `hobbit` and the class itself are completely unaffected, because nothing about assigning `dune.library_name` reached up into the class; it only ever wrote into `dune`'s own instance namespace.

### The mutable class attribute trap

This is worth a dedicated warning, because it's a genuine, common bug rather than a curiosity. Watch closely: 

In [ ]:
class Cart:
    items = []   # a class attribute, and it's a mutable list

    def add(self, item):
        self.items.append(item)

cart_a = Cart()
cart_b = Cart()

cart_a.add("apple")

print(cart_a.items)
print(cart_b.items)

`cart_b.items` shows `["apple"]` too — a customer's cart contains an item nobody added to it. Here's why: `items = []` created exactly *one* list, attached to the class. `self.items.append(...)` never assigns to `self.items` (which would create a new instance attribute, shadowing the class one, exactly as happened with `library_name`) — it looks up `self.items`, finds the *shared* class-level list (since neither cart has its own instance attribute named `items`), and mutates that shared list in place. Both carts were pointing at the same list the entire time, and `.append()` changed the one object both of them see.

The fix is to make each instance genuinely have its own list, set up fresh inside `__init__`: 

In [ ]:
class Cart:
    def __init__(self):
        self.items = []   # a new, independent list for every instance

    def add(self, item):
        self.items.append(item)

cart_a = Cart()
cart_b = Cart()

cart_a.add("apple")

print(cart_a.items)
print(cart_b.items)

The rule to take away: class attributes are the right tool for data that's genuinely meant to be shared and identical across every instance (a constant like `library_name`, which nothing ever mutates). The moment an attribute is something each instance should own and modify independently — especially if it's a mutable type like a list or dictionary — it belongs in `__init__`, attached through `self`, not sitting in the class body.

### One problem

> Write a class `Student` with a class attribute `school_name = "SUZH Academy"` and instance attributes `name` and `grades` (an empty list, correctly initialized per-instance). Add a method `add_grade(grade)`. Create two students, add different grades to each, and print both students' grades along with the shared school name.

In [ ]:
# TODO: define Student with correct instance-vs-class attribute handling


## 37. Instance Methods, `@classmethod`, and `@staticmethod`

Every method you've written so far has taken `self` as its first parameter — an **instance method**, operating on one specific object. There are two other kinds of method Python supports, and the difference between all three is really about *what each one receives automatically*, not an arbitrary style choice.

### A motivating problem

Suppose you want a way to build a `Book` from a single formatted string, like `"Dune, Frank Herbert, 412"`, as an alternative to the normal constructor.

In [ ]:
class Book:
    def __init__(self, title, author, pages):
        self.title = title
        self.author = author
        self.pages = pages

    def summary(self):
        return f"{self.title} by {self.author} ({self.pages} pages)"

    @classmethod
    def from_string(cls, text):
        title, author, pages = text.split(", ")
        return cls(title, author, int(pages))

dune = Book.from_string("Dune, Frank Herbert, 412")
print(dune.summary())

`from_string` doesn't take `self`, because there's no specific book yet to operate on — its whole job is to *create* one. Instead, it takes `cls`, which plays the same role for the class that `self` plays for an instance: Python automatically passes in the class itself (`Book`) as `cls`, the same mechanical way it passes an instance in as `self`. `cls(...)` inside the method then calls the class as a constructor, exactly as `Book(...)` would from outside — written as `cls` rather than hardcoding `Book` specifically, so that the method continues to work correctly even for a subclass, a benefit that becomes concrete once inheritance is introduced two chapters from now.

A `@classmethod` is most useful for exactly this pattern: an alternative way to construct an object, when the normal `__init__` shape doesn't fit every situation you need to support.

### `@staticmethod` — belonging to the class, needing neither `self` nor `cls`

Sometimes a piece of logic is thematically related to a class, but genuinely doesn't need any particular instance *or* the class itself to do its job.

In [ ]:
class Book:
    def __init__(self, title, pages):
        self.title = title
        self.pages = pages

    @staticmethod
    def pages_to_minutes(pages, pages_per_minute=1.5):
        return pages / pages_per_minute

dune = Book("Dune", 412)
estimated_minutes = Book.pages_to_minutes(dune.pages)
print(f"{estimated_minutes:.0f} minutes")

`pages_to_minutes` takes neither `self` nor `cls` — it's just an ordinary function that happens to live inside the class, grouped there because it's conceptually about books, even though it doesn't touch any specific book's or the class's own state. It could exist as a free-standing function outside the class and behave identically; putting it inside `Book` is purely an organizational choice, signaling "this belongs with books" to anyone reading the code.

### What each one is actually receiving

```text
instance method (self, ...)   → receives the specific object it was called on
classmethod    (cls, ...)     → receives the class itself
staticmethod   (...)          → receives neither; just a regular function, grouped by association
```

That's the entire distinction — not three arbitrary decorators to memorize, but three different answers to "what, if anything, does this method automatically receive." Choosing between them is really choosing what the method's logic actually needs: does it need one specific object's data (`self`), does it need to know or construct instances of the class in general (`cls`), or does it need neither, and just happen to belong conceptually with this class (neither)?

### One problem

> Add a `@classmethod` named `square(cls, side)` to the `Rectangle` class from Chapter 34, which returns a `Rectangle` with equal width and height. Also add a `@staticmethod` named `is_valid_dimension(value)` that returns whether a given number is greater than zero. Use both.

In [ ]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    # TODO: add square() as a classmethod and is_valid_dimension() as a staticmethod


## 38. Inheritance and `super()`

Suppose the library program needs a second kind of item — an audiobook, which is mostly like a book but tracks duration instead of page count, and has a slightly different summary. Writing it from scratch would duplicate everything `Book` already does.

In [ ]:
class Book:
    def __init__(self, title, author):
        self.title = title
        self.author = author

    def summary(self):
        return f"{self.title} by {self.author}"


class Audiobook(Book):
    def __init__(self, title, author, duration_minutes):
        super().__init__(title, author)
        self.duration_minutes = duration_minutes

    def summary(self):
        base = super().summary()
        return f"{base} ({self.duration_minutes} minutes)"


dune_audio = Audiobook("Dune", "Frank Herbert", 1260)
print(dune_audio.summary())

`class Audiobook(Book):` declares `Audiobook` as a **derived class** (or subclass), with `Book` as its **base class** (or superclass). Every `Audiobook` is also, automatically, a `Book` — it inherits every attribute and method `Book` defines, and then adds or changes what it needs to.

### `super()` is not simply "call the parent"

It's tempting to read `super().__init__(title, author)` as "call `Book`'s `__init__`," and for a class with exactly one base class, that reading gives the right answer. But it's the wrong *mental model* to build, because it breaks the moment more than one class is involved in the chain — which is exactly what Chapter 39 covers. The more accurate way to think about `super()`: it hands you a way to call the *next* method in line, following a specific, computed order of classes — called the **method resolution order** — rather than literally meaning "my direct parent." For a simple, single-inheritance case like `Audiobook(Book)`, that order happens to be just `Audiobook`, then `Book`, so the distinction is invisible right now. Hold onto it anyway; you'll need the precise version very soon.

### Overriding versus extending

`Audiobook.summary` **overrides** `Book.summary` — same method name, different behavior for `Audiobook` objects specifically. But notice it didn't rewrite the whole thing from scratch: `base = super().summary()` calls `Book`'s original version to get the shared part, and then builds on top of it. This is **extending**, not just overriding — reusing the base class's behavior rather than duplicating it, which is generally the better instinct whenever a subclass's version of a method genuinely does "everything the base version does, plus a bit more," rather than something completely unrelated.

### Checking the relationship

Two tools let you inspect this relationship directly, and both come up constantly in real code: 

In [ ]:
print(isinstance(dune_audio, Audiobook))
print(isinstance(dune_audio, Book))
print(issubclass(Audiobook, Book))

`isinstance(dune_audio, Book)` being `True` is worth sitting with: `dune_audio` was created as an `Audiobook`, never literally as a `Book`, yet it *is* one, in the sense that matters — anywhere code expects "something that behaves like a `Book`," an `Audiobook` genuinely qualifies, because it inherited everything a `Book` guarantees and only added to it.

### One problem

> Create a base class `Shape` with a method `area()` that returns `0` by default, and a subclass `Circle` that takes a `radius` in `__init__` and overrides `area()` to return the correct value (`3.14159 * radius ** 2`). Confirm `isinstance(your_circle, Shape)` is `True`.

In [ ]:
# TODO: define Shape and Circle as described


## 39. Multiple Inheritance and MRO

Python allows a class to inherit from more than one base class at once. This is a genuinely powerful feature, and it comes with a genuinely famous problem attached to it.

### The diamond

Consider four classes shaped like this: 

In [ ]:
class Animal:
    def speak(self):
        return "..."


class Dog(Animal):
    def speak(self):
        return "Woof"


class Cat(Animal):
    def speak(self):
        return "Meow"


class DogCatHybrid(Dog, Cat):
    pass


hybrid = DogCatHybrid()
print(hybrid.speak())

`DogCatHybrid` inherits from *both* `Dog` and `Cat`, each of which inherits from `Animal` — a diamond shape, if you draw the inheritance arrows:

```text
        Animal
       /      \
     Dog      Cat
       \      /
    DogCatHybrid
```

Both `Dog` and `Cat` define `speak`. When `hybrid.speak()` is called, which one runs? It printed `"Woof"` — but *why* `Dog`'s version rather than `Cat`'s isn't something to guess at; Python computes an exact, deterministic order and you can inspect it directly.

### `__mro__` — the actual, computed order

Every class has an `__mro__` attribute — its **method resolution order** — listing, in priority order, exactly where Python will look for an attribute or method.

In [ ]:
print(DogCatHybrid.__mro__)

Read left to right: `DogCatHybrid`, then `Dog`, then `Cat`, then `Animal`, then `object` (the base every class ultimately inherits from, whether written explicitly or not). `hybrid.speak()` searches this list in order and stops at the first match — `DogCatHybrid` itself has no `speak`, so it moves to `Dog`, finds one there, and stops before ever reaching `Cat`.

This is worth restating as the actual rule, replacing whatever intuitive guess you might have had: **Python does not search "the first parent, then dig all the way down that branch, then try the next parent."** It computes one single, flattened order for the whole class up front — using an algorithm called C3 linearization — and that order is what every attribute lookup follows, without exception.

### What C3 linearization is actually trying to guarantee

The full algorithm is more mechanical detail than a beginner needs, but the *guarantee* it provides is worth understanding, because it explains why the order came out the way it did: a class always appears before its own base classes in the MRO, and — this is the part that resolves the diamond correctly — the order you listed the base classes in `class DogCatHybrid(Dog, Cat):` is respected: `Dog` was listed first, so `Dog` (and anything defined only on `Dog`) takes priority over `Cat` in the search order. Reversing the declaration reverses the outcome: 

In [ ]:
class CatDogHybrid(Cat, Dog):
    pass

reversed_hybrid = CatDogHybrid()
print(reversed_hybrid.speak())
print(CatDogHybrid.__mro__)

Same two base classes, listed in the opposite order, and `Cat` now wins. The MRO isn't arbitrary or Python "picking a side" — it directly reflects the order you wrote the base classes in, made precise and consistent even in more tangled diamond shapes than this simple example.

### `super()` in a multiple-inheritance chain — the payoff of Chapter 38's warning

This is exactly why the last chapter insisted `super()` means "the next class in the MRO," not "my parent." In a single-inheritance chain, those two descriptions happen to agree. In a diamond, they don't — "my parent" is ambiguous (which one?), but "the next class in the MRO" is always a single, well-defined answer, because the MRO is one flat, ordered list, regardless of how many base classes are actually involved.

### One problem

> Given `class A`, `class B(A)`, `class C(A)`, and `class D(B, C)`, each defining a method `identify()` that returns its own class name (except `A`, define it too, for a base case), print `D().identify()` and `D.__mro__`, and explain — in a comment — why the result is what it is.

In [ ]:
# TODO: define A, B, C, D as described, then print D().identify() and D.__mro__


## 40. Polymorphism and Duck Typing

Look at this function, and notice what it doesn't check: 

In [ ]:
def announce(shape):
    print(f"This shape has an area of {shape.area()}")


class Circle:
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return 3.14159 * self.radius ** 2


class Square:
    def __init__(self, side):
        self.side = side

    def area(self):
        return self.side ** 2


announce(Circle(3))
announce(Square(4))

`announce` never asks what type `shape` is. It calls `shape.area()` and trusts that whatever it was handed has an `area` method that behaves sensibly. `Circle` and `Square` share no inheritance relationship at all here — neither one is a subclass of the other, or of some common `Shape` base — and `announce` still works correctly with both, because both happen to support the one operation it actually needs.

This is **polymorphism** — literally "many forms": the same piece of code (`announce`) working correctly across genuinely different types, as long as each type supports the specific behavior that code relies on. Chapter 38 showed one route to polymorphism, through inheritance and overriding — a `Dog` and a `Cat`, related through a shared `Animal` base, both responding sensibly to `.speak()`. This chapter shows that inheritance was never a requirement for it.

### Duck typing

The specific style shown above — no shared base class, no formal contract, just "does it have the method I need" — has a name that's become genuinely standard vocabulary: **duck typing**, from the saying "if it walks like a duck and quacks like a duck, it's a duck." Python doesn't check `shape`'s type before calling `.area()`; it just tries the call, and if the object supports it, everything works, regardless of what family tree that object belongs to.

In [ ]:
class WeirdNotAShape:
    def area(self):
        return 999

announce(WeirdNotAShape())   # still works -- announce never checked the type

`WeirdNotAShape` has nothing to do with `Circle` or `Square` conceptually — it just happens to expose an `.area()` method, and that's the entire bar `announce` set. Whether this is a feature or a risk depends on context, and it's worth being honest about both sides rather than treating duck typing as an unqualified good.

### Why Python leans this way

Contrast this with a more rigid alternative: `announce` could have demanded `isinstance(shape, Shape)` for some formally declared `Shape` base class, rejecting anything that wasn't explicitly related. That approach *guarantees* more — you know for certain any accepted object was deliberately designed to be a shape — but it also *requires* more: every type you ever want to pass in must be written (or rewritten) to inherit from `Shape`, even if it already, structurally, does everything needed.

Python's culture generally favors the looser version: judge an object by what it can *do*, not by what family it was born into. This connects to something you may have run across as a saying about Python's design philosophy — code should generally ask forgiveness (try the operation, handle it if it fails) rather than permission (check the type defensively before ever attempting anything) — and duck typing is that same instinct applied to types specifically.

### Where it can go wrong

The cost is that failures move later and become less specific. Pass something without an `.area()` method, and you don't find out until the exact line that calls it: 

In [ ]:
announce("just a string")

No warning at the point `announce` was called — the error only surfaces once Python actually tries `shape.area()` and discovers strings don't have that method. In a large program, that failure could be far removed from the actual mistake (passing the wrong kind of value in the first place), which is one of the real motivations behind the typing tools coming later in this book — `Protocol`, in particular, in Chapter 60, lets you state "anything with an `.area()` method is acceptable here" in a way tools can check *before* the code ever runs, without forcing formal inheritance either.

### One problem

> Write a function `total_cost(items)` that takes a list of objects, each of which has a `.price()` method, and returns the sum of calling `.price()` on every item. Create two unrelated classes (no shared base class) that each define `.price()` differently, and confirm `total_cost` works on a mixed list of both.

In [ ]:
# TODO: define two unrelated classes with .price(), and total_cost()


## 41. Abstract Base Classes

The last chapter's duck typing worked, but it also quietly allowed `WeirdNotAShape` and even a bare string to be passed into `announce` without complaint until the exact moment of failure. Sometimes you want the opposite guarantee: a way to say, explicitly and enforceably, "any class claiming to be a `Shape` *must* provide an `area()` method, or Python should refuse to even create an instance of it." That's what an **abstract base class** provides.

In [ ]:
from abc import ABC, abstractmethod


class Shape(ABC):
    @abstractmethod
    def area(self):
        ...


class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return 3.14159 * self.radius ** 2


c = Circle(3)
print(c.area())

`Shape` inherits from `ABC` (short for Abstract Base Class), and `area` is marked `@abstractmethod` — a declaration that says "every concrete subclass must provide this," without providing an actual implementation itself (the `...` body is never meant to run). Now watch what happens if a subclass forgets to implement it: 

In [ ]:
class BrokenShape(Shape):
    pass

BrokenShape()

`TypeError`, and it happens the moment you try to *create* a `BrokenShape`, not later when something tries to call a missing method. That's the real value an ABC adds over plain duck typing: the mistake is caught immediately, at the earliest possible point, with a specific and honest error message, rather than surfacing much later as an `AttributeError` deep inside some unrelated function that happened to call `.area()`.

### Abstract versus concrete

`Shape` is **abstract** — it can never be instantiated directly, precisely because it has at least one unimplemented `abstractmethod`.

In [ ]:
Shape()

That fails too, for the same underlying reason `BrokenShape()` did — `Shape` itself never provided a real `area()`, so Python won't let you create a bare `Shape` at all. `Circle` is **concrete** — every abstract method has a real implementation, so it can be instantiated normally.

An abstract base class, in other words, defines a *contract* — "anything calling itself a `Shape` must be able to do this" — and Python enforces that contract at object-creation time, which plain inheritance (Chapter 38) and plain duck typing (Chapter 40) both leave entirely to convention and hope.

### When this is worth reaching for

An ABC is the right tool when you're deliberately designing a family of related types around a shared contract, and you want that contract enforced rather than merely documented — a set of payment processors that must all support `.charge(amount)`, for instance, where forgetting to implement it should be a loud, immediate error rather than a bug discovered in production. It's the *wrong* tool for the kind of loose, incidental sharing `announce()` handled fine in the last chapter — forcing every type that merely happens to have an `.area()` method to formally inherit from a common `Shape` just to satisfy a design pattern adds ceremony without adding real value, especially for types you don't control or can't modify.

The two tools aren't in competition so much as suited to different situations: duck typing when behavior matters more than lineage and flexibility is the priority; an ABC when you're the one defining a family of types and want Python itself to enforce that every member honors the contract.

### One problem

> Define an abstract base class `PaymentMethod` with an abstract method `charge(amount)`. Create two concrete subclasses, `CreditCard` and `Cash`, each implementing `charge` differently (e.g., printing a different message). Confirm that trying to instantiate `PaymentMethod` directly raises an error.

In [ ]:
from abc import ABC, abstractmethod

# TODO: define PaymentMethod, CreditCard, and Cash as described


## 42. Dunder Methods

Consider two `Book` objects and something that looks like it should obviously fail: 

In [ ]:
class Book:
    def __init__(self, title, pages):
        self.title = title
        self.pages = pages


b1 = Book("Dune", 412)
print(b1)

`<__main__.Book object at 0x...>` — technically correct, useless to a person. Now compare a built-in type: 

In [ ]:
print([1, 2, 3])
print("hello")

Lists and strings print something meaningful. The difference isn't that Python treats built-in types specially at the language level — it's that `list` and `str` each define a method telling `print` exactly how to represent them as text, and `Book`, as written above, doesn't. You can give `Book` the same ability.

In [ ]:
class Book:
    def __init__(self, title, pages):
        self.title = title
        self.pages = pages

    def __str__(self):
        return f"{self.title} ({self.pages} pages)"


b1 = Book("Dune", 412)
print(b1)

### The idea underneath this: a protocol for common operations

`__str__` is one member of a large family of methods with double underscores on both sides — often called **dunder methods**, short for "double underscore." The pattern to understand isn't any one of them individually; it's what they all have in common: Python defines a fixed set of operations — printing, equality, comparison, addition, length, indexing — and instead of hardcoding how each one works for every possible type, it asks each object, through a specifically named method, how *that type* wants the operation to behave. `print(x)` doesn't know what `x` is; it looks for `x.__str__()` and uses whatever that returns.

```text
print(obj)
    ↓
looks for obj.__str__()
    ↓
uses whatever string that returns
```

This is the same underlying idea as the iterator protocol from Chapter 33 (`iter()` looking for a specific method) and duck typing from Chapter 40 (behavior determined by what an object supports) — a consistent theme in Python's design: define a small set of well-known method names, and let any type participate in a built-in operation simply by implementing the right one.

### `__str__` versus `__repr__`

Python actually has two related methods here, for two different audiences.

In [ ]:
class Book:
    def __init__(self, title, pages):
        self.title = title
        self.pages = pages

    def __str__(self):
        return f"{self.title} ({self.pages} pages)"

    def __repr__(self):
        return f"Book(title={self.title!r}, pages={self.pages!r})"


b1 = Book("Dune", 412)
print(b1)          # uses __str__
print([b1])        # a list of books uses __repr__ for each item
b1                 # bare value in a notebook cell also uses __repr__

`__str__` is meant for a person — readable, friendly. `__repr__` is meant to be unambiguous, ideally something that looks like valid Python that could recreate the object — useful when debugging, or when an object shows up nested inside another structure like a list, which is why `[b1]` used `__repr__` rather than `__str__`. If `__str__` is missing, Python falls back to `__repr__` for both purposes — which is why defining at least `__repr__` is generally considered worthwhile even for simple classes.

### Comparison and arithmetic follow the exact same idea

`==`, `<`, and `+` between two of your own objects don't do anything meaningful until you tell Python how, through the corresponding dunder method.

In [ ]:
class Book:
    def __init__(self, title, pages):
        self.title = title
        self.pages = pages

    def __eq__(self, other):
        return self.pages == other.pages

    def __lt__(self, other):
        return self.pages < other.pages

    def __add__(self, other):
        return self.pages + other.pages


dune = Book("Dune", 412)
hobbit = Book("The Hobbit", 310)

print(dune == hobbit)
print(dune < hobbit)
print(dune + hobbit)

`dune == hobbit` triggers `dune.__eq__(hobbit)`; `dune < hobbit` triggers `dune.__lt__(hobbit)`; `dune + hobbit` triggers `dune.__add__(hobbit)`. Every one of these is just an ordinary method with a special name that Python's built-in operators know to check for — `a + b`, at the language level, essentially *is* `a.__add__(b)` when `a` is your own object, the same mechanical "operator becomes a method call" translation for every one of these symbols.

### `__len__` and `__getitem__` — letting your object behave like a sequence

Two more, briefly, to reinforce that this is a protocol you can opt into as far as makes sense for your type — not an all-or-nothing package.

In [ ]:
class Playlist:
    def __init__(self, songs):
        self.songs = songs

    def __len__(self):
        return len(self.songs)

    def __getitem__(self, index):
        return self.songs[index]


p = Playlist(["Song A", "Song B", "Song C"])
print(len(p))
print(p[1])

`len(p)` triggers `p.__len__()`; `p[1]` triggers `p.__getitem__(1)`. Implement both, and Python's built-in `len()` and `[]` syntax work on your object exactly as they would on a list — and, as a genuinely useful side effect, `__getitem__` alone is enough to make an object iterable with a plain `for` loop too, since `for` can fall back to repeatedly indexing with `0, 1, 2, ...` when no proper `__iter__` is defined (the cleaner, more deliberate way to make something iterable is the actual iterator protocol from Chapter 44, but it's worth knowing this fallback exists).

### The point of this chapter, stated directly

The goal was never to memorize a dictionary of magic method names. It's to recognize the *pattern*: Python's built-in operators and functions are, for your own types, really just polite requests routed to specific, predictably named methods — implement the ones relevant to your type, skip the ones that don't make sense for it, and your objects integrate naturally with the rest of the language rather than needing special-case handling anywhere.

### One problem

> Write a class `Money` with an amount and a currency, implementing `__str__` (something like `"$50"`), `__eq__` (equal if amount and currency both match), and `__add__` (returns a new `Money` with amounts summed, only if currencies match — raise a `ValueError` otherwise).

In [ ]:
# TODO: define Money with __str__, __eq__, and __add__


## 43. Properties

Here's a class where something could quietly go wrong: 

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius


t = Temperature(25)
t.celsius = -500   # physically impossible, and nothing stops it
print(t.celsius)

`celsius` is a plain attribute — direct access, no checks, no matter what gets assigned to it. Absolute zero is around `-273.15°C`; nothing in this class knows or cares that `-500` is nonsense.

You might reach for a method instead: 

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.set_celsius(celsius)

    def set_celsius(self, value):
        if value < -273.15:
            raise ValueError("Temperature below absolute zero")
        self._celsius = value

    def get_celsius(self):
        return self._celsius


t = Temperature(25)
t.set_celsius(-500)

This actually works — the validation runs, the bad value is rejected. But look at the cost: every read and write now goes through `get_celsius()` and `set_celsius()` explicitly, instead of the plain `t.celsius` syntax from before. Anyone (including code you wrote earlier, using the plain-attribute version) that expected `t.celsius` to just work needs to be rewritten. That's a real problem in a growing codebase — changing *how* an attribute is stored shouldn't force every single place that *uses* it to change too.

### `@property` — validation, with the plain-attribute syntax preserved

A property lets a method be *accessed* exactly like an attribute, while still running real code behind the scenes.

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius   # this line already uses the property, below

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("Temperature below absolute zero")
        self._celsius = value


t = Temperature(25)
print(t.celsius)

t.celsius = -500

`t.celsius` — no parentheses, looks exactly like plain attribute access — actually calls the method decorated with `@property`. `t.celsius = -500` — also looks like plain attribute assignment — actually calls the method decorated with `@celsius.setter`, and that method is free to reject the value, exactly as `set_celsius` did, without forcing callers to write `t.set_celsius(-500)` instead of the natural `t.celsius = -500`.

Notice the underlying storage moved to `self._celsius` — a leading underscore, by convention, meaning "this is an implementation detail; use the property, not this directly." Python doesn't enforce that convention the way some languages enforce genuine privacy, but it's a widely understood signal, and the property is precisely what makes it safe to follow: nobody needs to touch `_celsius` directly, because `celsius` (no underscore) already gives them exactly the interface they want.

### The actual point: the interface stays stable even as the implementation changes

This is the real reason properties matter, beyond validation specifically. Suppose `Temperature` originally stored only Celsius, and later needs to also support Fahrenheit, computed on demand rather than stored separately: 

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("Temperature below absolute zero")
        self._celsius = value

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32


t = Temperature(25)
print(t.fahrenheit)

`fahrenheit` has no setter here, deliberately — reading `t.fahrenheit` computes it fresh from `_celsius` every time, but `t.fahrenheit = 100` would fail, since there's no `@fahrenheit.setter` to catch it. That's a **read-only property**: something that looks like an attribute from the outside, is entirely computed rather than stored, and can't be assigned to directly. Every piece of code using `t.celsius` or `t.fahrenheit` never needed to know or care whether either one is stored directly or computed on the fly — the property's whole purpose is making that an invisible implementation detail from the outside.

### One problem

> Write a class `BankAccount` with a `_balance` (starting at `0`) and a read-only property `balance` that returns it, plus methods `deposit(amount)` and `withdraw(amount)` — `deposit` should reject negative amounts, and `withdraw` should reject amounts greater than the current balance, both by raising `ValueError`.

In [ ]:
# TODO: define BankAccount as described


# Part III — Iteration, Generators and Lazy Computation

## 44. Iterators in Depth

Chapter 33 walked through `iter()` and `next()` as the mechanism underneath every `for` loop, using a built-in string as the example. Now that classes and dunder methods are both available, it's worth actually building one, because doing it once by hand makes the protocol permanently concrete rather than something you take on faith.

### The iterator protocol, precisely

An object is an iterator if it implements two specific dunder methods:

```text
__iter__(self)   → returns the iterator itself
__next__(self)   → returns the next value, or raises StopIteration when done
```

That's the whole contract. `iter(obj)` calls `obj.__iter__()`; `next(obj)` calls `obj.__next__()`. Here's a small, complete iterator that counts upward from a starting value, stopping before a limit — deliberately reimplementing a piece of what `range()` already does, specifically so the comparison to something familiar is available.

In [ ]:
class CountUpTo:
    def __init__(self, limit):
        self.limit = limit
        self.current = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.current >= self.limit:
            raise StopIteration
        value = self.current
        self.current += 1
        return value


counter = CountUpTo(5)
print(next(counter))
print(next(counter))
print(next(counter))

Each `next()` call reads `self.current`, checks it against `self.limit`, and either raises `StopIteration` or advances and returns a value. Because the object tracks its own progress in `self.current`, calling `next()` repeatedly genuinely moves forward each time — the exact behavior you manually drove with `iter()`/`next()` in Chapter 33, now written by hand instead of borrowed from a string.

### Why `__iter__` just returns `self`

`__iter__` exists because `iter()` needs to work on *any* iterable, including plain collections that aren't themselves iterators — a list's `__iter__` returns a fresh, separate iterator object, so the list itself can be iterated over repeatedly (exactly the distinction Chapter 33 drew between iterable and iterator). `CountUpTo`, here, is deliberately built to *be* its own iterator — it has no separate underlying collection to hand out iterators over, it just directly counts. Returning `self` from `__iter__` is the correct choice specifically for an object that already tracks its own iteration state.

### Using it with `for`, exactly like anything else iterable

Because `CountUpTo` correctly implements the protocol, it plugs directly into every piece of syntax that expects an iterable, without any special-case code on Python's side.

In [ ]:
for n in CountUpTo(5):
    print(n)

In [ ]:
print(list(CountUpTo(4)))
print(sum(CountUpTo(10)))

`for`, `list()`, and `sum()` never needed to know anything about `CountUpTo` specifically. Each of them is written once, generically, against the iterator protocol — call `iter()`, then call `next()` until `StopIteration` — and any object honoring that protocol, yours included, works with all of them for free. This is the same payoff Chapter 40 described for duck typing in general, now demonstrated concretely for one specific, extremely common protocol.

### One caveat worth remembering from Chapter 33

Because `CountUpTo(5)` returns itself from `__iter__`, a single instance behaves like an iterator, not a re-iterable collection — once exhausted, it stays exhausted.

In [ ]:
c = CountUpTo(3)
print(list(c))
print(list(c))   # already exhausted -- nothing left

To loop over the same range of values twice, you need a *new* `CountUpTo(3)` instance each time — exactly as a fresh call to `iter()` on a list gives you a fresh iterator, but reusing an already-consumed iterator gives you nothing further.

### One problem

> Write an iterator class `EvenNumbers(limit)` that yields even numbers starting from `0`, stopping before `limit`. Confirm it works with both a `for` loop and `list()`.

In [ ]:
# TODO: define EvenNumbers implementing __iter__ and __next__


## 45. Generators and `yield`

The `CountUpTo` class from the last chapter works, but notice how much bookkeeping it required for a fairly simple idea: an `__init__` to store the state, a `__next__` to advance it, an explicit `StopIteration`. A **generator** lets you write the exact same behavior as what looks like an ordinary function, with almost none of that machinery.

In [ ]:
def count_up_to(limit):
    current = 0
    while current < limit:
        yield current
        current += 1


for n in count_up_to(5):
    print(n)

That's the entire implementation — five lines, doing what took a full class with two dunder methods in the previous chapter. `yield` is the new piece, and it's worth being precise that it is genuinely not "return, but repeatable" — it behaves in a way ordinary `return` never does, and the difference is worth tracing carefully rather than accepting on faith.

### What calling a generator function actually does

Here's the detail that surprises almost everyone the first time: 

In [ ]:
result = count_up_to(3)
print(result)
print(type(result))

Calling `count_up_to(3)` did **not** run any of the function's body — no loop iteration happened, nothing was computed. What came back is a **generator object**: a ready-to-run, paused-at-the-very-start iterator, automatically satisfying the exact protocol from the last chapter (`__iter__` and `__next__`), without you writing either method yourself. The function's body only starts executing once something actually asks it for a value.

In [ ]:
gen = count_up_to(3)

print(next(gen))
print(next(gen))
print(next(gen))
print(next(gen))

Trace this against the function's source line by line. The first `next(gen)` starts the function running from the top: `current = 0`, enter the `while` loop (`0 < 3` is true), hit `yield current` — and this is the part that makes generators different from every function you've written before. `yield` doesn't just produce a value the way `return` does; it **pauses** the function's execution entirely, right at that line, remembering everything about its state — the value of `current`, the fact that it's partway through the `while` loop's body — and hands `0` back to whoever called `next()`.

The *second* `next(gen)` doesn't start the function over. It **resumes** execution from exactly the paused `yield` line, continues to `current += 1` (making `current` become `1`), loops back to check `1 < 3`, hits `yield current` again, and pauses again, handing back `1`. This repeats until `current < limit` becomes false, at which point the function reaches its natural end — and only then does it raise `StopIteration`, automatically, the same signal Chapter 44 required you to raise by hand.

```text
call count_up_to(3)
        ↓
generator object created (nothing has run yet)
        ↓
next(gen)  →  runs until the first yield  →  pauses, returns a value
        ↓
next(gen)  →  resumes right after that yield  →  runs until the next yield  →  pauses again
        ↓
      ...repeats...
        ↓
function body finishes naturally  →  StopIteration
```

### Why "pausing and resuming" is a genuinely new kind of behavior

Every ordinary function you've written starts fresh at the top on every call, and any local variables it had are gone the moment it returns — Chapter 24 established this explicitly. A generator function breaks that rule on purpose: calling it doesn't run it to completion, and its local state (`current`, here) survives *between* calls to `next()`, exactly where it left off. This is what "lazy evaluation" means in practice: values are produced one at a time, only when actually requested, rather than an entire result being computed and handed over all at once.

### The memory consequence — why this matters beyond being a neat trick

Compare a version that builds a full list up front: 

In [ ]:
def count_up_to_list(limit):
    result = []
    current = 0
    while current < limit:
        result.append(current)
        current += 1
    return result


big_list = count_up_to_list(1_000_000)
big_generator = count_up_to(1_000_000)

print(type(big_list), type(big_generator))

`count_up_to_list(1_000_000)` genuinely builds and holds a million-element list in memory, all at once, before you ever look at a single value. `count_up_to(1_000_000)` builds nothing yet at all — it's a paused function, holding only the small amount of state needed to produce the *next* value whenever asked, one at a time. If you only ever need to process values one at a time — printing them, summing them, stopping early partway through — the generator does the same job using a small, constant amount of memory, regardless of how large the limit is, while the list version's memory use grows directly with it.

### One problem

> Write a generator function `fibonacci(n)` that yields the first `n` Fibonacci numbers (`0, 1, 1, 2, 3, 5, ...`) one at a time, without ever building a list.

In [ ]:
# TODO: define fibonacci(n) as a generator function


## 46. Generator Expressions vs List Comprehensions

Chapter 31 introduced this: 

In [ ]:
numbers = [1, 2, 3, 4, 5]
squares = [n ** 2 for n in numbers]
print(squares)

Swap the square brackets for parentheses, and the syntax stays almost identical, but the result is a completely different kind of thing: 

In [ ]:
squares_gen = (n ** 2 for n in numbers)
print(squares_gen)
print(type(squares_gen))

A **generator expression** — the same expression-plus-`for` shape as a list comprehension, but producing a generator object instead of a list, exactly the kind of object the last chapter spent its entire length explaining. Nothing was computed yet; each `n ** 2` only gets evaluated the moment something actually asks the generator for its next value.

In [ ]:
squares_gen = (n ** 2 for n in numbers)

for value in squares_gen:
    print(value)

### Eager versus lazy, side by side

The list comprehension is **eager**: the entire list — every single squared value — is computed and stored in memory the instant that line runs, whether or not you ever look at all of it. The generator expression is **lazy**: nothing is computed until it's asked for, one value at a time, exactly following Chapter 45's pause-and-resume model.

This distinction is invisible for a five-element list, and starts mattering the moment the input is large or the downstream code doesn't need every value: 

In [ ]:
numbers = range(10_000_000)

# Building the full list uses real, immediate memory for ten million squared values.
# squares_list = [n ** 2 for n in numbers]

# The generator uses a tiny, constant amount of memory, computing lazily.
squares_gen = (n ** 2 for n in numbers)

# Only three values are ever actually computed here.
for i, value in enumerate(squares_gen):
    if i >= 3:
        break
    print(value)

The commented-out list version, if actually run, would compute and store all ten million squares immediately, regardless of the fact that only three were ever going to be used. The generator version computes exactly three values and then simply stops being asked for more — the remaining 9,999,997 are never generated at all.

### When each is the right choice

A list comprehension is right when you need the whole collection at once — to index into it repeatedly, pass it somewhere expecting a list specifically, or iterate over it more than once (a generator, once consumed, is exhausted, exactly as Chapter 45's `count_up_to` was after being fully drained). A generator expression is right when you're processing a sequence of values once, in order, especially when the full collection would be large or you might stop early — summing values, searching for the first match, feeding values one at a time into another piece of code that only needs to see them once.

In [ ]:
# sum() only needs to see each value once, in order -- a natural fit for a generator
total = sum(n ** 2 for n in range(1, 1_000_001))
print(total)

Notice the parentheses around the generator expression were dropped entirely there — when a generator expression is the *only* argument to a function call, Python lets you skip the extra pair of parentheses, since the call's own parentheses already do the job.

### One problem

> Given a very large `range(1, 10_000_001)`, use a generator expression together with `sum()` to compute the sum of every number in that range that's divisible by 7, without ever constructing a full list of them.

In [ ]:
# TODO: compute the sum using a generator expression, not a list comprehension


## 47. Closures and Nested Functions

You already know functions can be defined inside other functions — nothing about `def` restricts where it can appear. Here's what that actually enables: 

In [ ]:
def make_greeter(greeting):
    def greet(name):
        return f"{greeting}, {name}!"
    return greet


hello_greeter = make_greeter("Hello")
print(hello_greeter("Ali"))

hi_greeter = make_greeter("Hi")
print(hi_greeter("Sara"))

`make_greeter` returns `greet` — an actual function object, not the result of calling it — and whatever it returns is then callable on its own, later, completely independent of `make_greeter`. That much follows from something you already know: functions are values, and returning one is no different in principle from returning a number or a string.

What's new, and worth stopping on, is this: `greet`'s body refers to `greeting`, a name that belongs to `make_greeter`, not to `greet` itself. By the time `hello_greeter("Ali")` actually runs, `make_greeter("Hello")` has long since finished and returned — by the scoping rules from Chapter 24, `greeting` should be gone, a purely local variable of a function call that's already over. And yet it clearly isn't gone; `hello_greeter` still remembers `"Hello"` specifically, and `hi_greeter` separately remembers `"Hi"`.

### What's actually happening

This is called a **closure**: `greet` doesn't just remember the *code* of its body — it carries along a reference to the specific variables from its **enclosing scope** (`make_greeter`'s local scope) that its body actually uses. Each call to `make_greeter` creates a genuinely separate `greeting` variable, and each function `make_greeter` returns closes over its *own* copy of that variable, independent of any other call's.

```text
make_greeter("Hello")
        ↓
creates a local 'greeting' = "Hello"
        ↓
defines greet, whose body references 'greeting'
        ↓
returns greet -- and greet keeps a live connection to that specific 'greeting'
```

You can inspect this connection directly — it's a real, named thing, not just a metaphor: 

In [ ]:
print(hello_greeter.__closure__[0].cell_contents)
print(hi_greeter.__closure__[0].cell_contents)

Two separate `greet` functions, two separate captured `greeting` values, exactly matching what each call to `make_greeter` was given.

### A closure that changes over time — and why it needs `nonlocal`

The example so far only *read* the captured variable. Try to update it from inside the nested function, the same way you would with an ordinary variable: 

In [ ]:
def make_counter():
    count = 0

    def increment():
        count = count + 1   # looks reasonable -- but watch closely
        return count

    return increment


counter = make_counter()
print(counter())
print(counter())

Both calls print `1`. Not `1`, then `2` — every single call resets to `1`. This is the exact shadowing behavior from Chapter 24, now appearing in a closure: `count = count + 1` inside `increment` contains an *assignment* to `count`, and just as in Chapter 24, the presence of that assignment anywhere in the function makes Python treat `count` as a brand-new local variable for the entire call — not the enclosing `count` from `make_counter` at all. `increment` never actually gets to *read* the outer `count` on the right-hand side either, because Python already decided, before running anything, that `count` is local here.

`nonlocal` is the tool that fixes this — the direct counterpart to `global` from Chapter 24, but for an enclosing function's scope specifically rather than the module-level global scope.

In [ ]:
def make_counter():
    count = 0

    def increment():
        nonlocal count
        count = count + 1
        return count

    return increment


counter = make_counter()
print(counter())
print(counter())
print(counter())

`nonlocal count` tells Python explicitly: "the `count` this function assigns to is not a new local variable — it's the one from the enclosing scope, and assignments here should actually modify that one." Now each call genuinely updates the same captured `count`, and a second, independent counter created separately doesn't interfere with it at all: 

In [ ]:
another_counter = make_counter()
print(another_counter())
print(counter())   # unaffected by the other counter's calls

### Why this matters, beyond being a clever trick

A closure gives you a function that carries its own private, persistent state, without needing a class at all — `make_counter()` produces something behaviorally similar to a tiny object with one hidden attribute and one method, using only functions. This is worth remembering as a genuine alternative tool, not just a stepping stone — and it's also the exact mechanism the next chapter depends on completely, since a decorator is, underneath everything, a closure that wraps another function.

### One problem

> Write a function `make_multiplier(factor)` that returns a function which multiplies its argument by `factor`. Create `double = make_multiplier(2)` and `triple = make_multiplier(3)`, and confirm they behave independently.

In [ ]:
# TODO: define make_multiplier as described


## 48. Decorators

Suppose several functions in a program each need to print how long they took to run. Writing that timing logic inside every single function would duplicate it everywhere, and the last chapter already showed the tool for wrapping one function's behavior around another without duplicating code: a closure that takes a function in, and returns a new function.

In [ ]:
import time


def apply_timer(func):
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        end = time.perf_counter()
        print(f"{func.__name__} took {end - start:.4f} seconds")
        return result
    return wrapper


def slow_function():
    total = 0
    for n in range(1_000_000):
        total += n
    return total


timed_slow_function = apply_timer(slow_function)
print(timed_slow_function())

`apply_timer` takes a function, `func`, as its argument — nothing new here; Chapter 22 already established that functions are ordinary values that can be passed around. It defines `wrapper`, a nested function that calls `func`, measuring the time around that call, and returns `wrapper` rather than calling it. `timed_slow_function` is now a *new* function — genuinely a closure, exactly as in the last chapter, having captured `func` (which is `slow_function`) from its enclosing scope — that does everything `slow_function` did, plus timing, without a single line of `slow_function`'s own code needing to change.

### Replacing the name, instead of creating a new one

Right now, calling this requires remembering to use `timed_slow_function` instead of `slow_function` everywhere. It's more useful to simply replace what the name `slow_function` refers to: 

In [ ]:
def slow_function():
    total = 0
    for n in range(1_000_000):
        total += n
    return total


slow_function = apply_timer(slow_function)
print(slow_function())

Nothing conceptually new happened — `slow_function` is just a name, and Chapter 4 already established that a name can be reassigned to point at a different value entirely, including a function value. After this reassignment, calling `slow_function()` actually runs `wrapper`, which in turn calls the *original* function (still reachable through `wrapper`'s closure, even though the name `slow_function` no longer points directly at it).

### `@decorator` is exactly this pattern, with dedicated syntax

The pattern "define a function, then immediately reassign its name to the result of wrapping it" is common enough that Python has syntax specifically for it: 

In [ ]:
@apply_timer
def slow_function():
    total = 0
    for n in range(1_000_000):
        total += n
    return total


print(slow_function())

`@apply_timer` directly above `def slow_function():` means exactly, and only, this:

```text
def slow_function():
    ...
slow_function = apply_timer(slow_function)
```

That's the entire meaning of `@` — take the function defined immediately below it, pass it into the named decorator function, and rebind the original name to whatever comes back. It's worth stating this precisely because `@decorator` can look, from the outside, like special language magic; it isn't — it's the closure pattern from the previous two examples, with syntax that saves you from writing the reassignment line yourself, applied to a function object using tools you already fully understand.

### A cost of this approach worth noticing directly

Check what `slow_function.__name__` reports, now that it's decorated: 

In [ ]:
print(slow_function.__name__)

`wrapper` — not `slow_function`. This makes sense mechanically (the name `slow_function` now refers to the `wrapper` function object, and `wrapper.__name__` is genuinely `"wrapper"`), but it's a real problem for debugging and introspection: anyone inspecting `slow_function` after decoration sees the wrapper's identity, not the original function's, and this gets worse the more decorators get applied. The next chapter's `functools.wraps` exists specifically to fix this.

### Decorators that take arguments themselves

Sometimes a decorator needs to be configurable — repeating a function's result a given number of times, say, rather than fixed timing behavior. This requires one more layer of nesting: a function that returns a decorator, rather than being a decorator directly.

In [ ]:
def repeat(times):
    def decorator(func):
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator


@repeat(times=3)
def say_hello(name):
    print(f"Hello, {name}")


say_hello("Ali")

`@repeat(times=3)` first calls `repeat(times=3)`, which returns `decorator` — and *that* is what actually gets applied to `say_hello`, exactly as `apply_timer` was applied directly above. `repeat` itself isn't a decorator; it's a function that *produces* one, configured with whatever argument it was given. Note also `wrapper(*args, **kwargs)` here, reusing the unpacking syntax from Chapter 32 — this lets `wrapper` accept whatever arguments the decorated function needs, without `apply_timer` or `repeat` needing to know in advance what those are.

### Stacking decorators

More than one decorator can be applied to the same function, and they apply from the bottom upward — the one closest to the `def` wraps first.

In [ ]:
@apply_timer
@repeat(times=2)
def greet_twice(name):
    print(f"Hi, {name}")


greet_twice("Sara")

Read bottom to top: `greet_twice` is first wrapped by `repeat(times=2)`'s decorator, producing a function that calls the original twice; *that* combined function is then wrapped by `apply_timer`, so the timing measures the entire two-call run as one unit. Order matters here exactly as it would if you wrote the equivalent nested calls by hand: `apply_timer(repeat(times=2)(greet_twice))`.

### One problem

> Write a decorator `log_calls` that prints the function's name and its arguments every time it's called, then calls and returns the original function's result unchanged. Apply it to any function of your choice that takes at least one argument.

In [ ]:
# TODO: define log_calls and apply it with @


## 49. `functools`

The last chapter ended with a real, visible problem: a decorated function loses its own identity — `__name__`, and along with it things like its docstring — because the name now points at `wrapper` instead. `functools.wraps` fixes exactly this.

In [ ]:
from functools import wraps


def apply_timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result
    return wrapper


@apply_timer
def slow_function():
    '''A function that pretends to be slow.'''
    return sum(range(1_000_000))


print(slow_function.__name__)
print(slow_function.__doc__)

`@wraps(func)` is, itself, a decorator — applied to `wrapper`, inside `apply_timer` — that copies `func`'s `__name__`, `__doc__`, and a few other identifying details onto `wrapper` after it's created. It doesn't change `wrapper`'s actual behavior at all; it only repairs the bookkeeping so that anything inspecting the decorated function sees the *original* function's identity, not the wrapper's. The practical rule: any decorator you write that defines an inner `wrapper` function should almost always apply `@wraps(func)` to it, as a matter of habit, the same way you'd habitually close a file.

### `lru_cache` — remembering results instead of recomputing them

Consider a function that's genuinely expensive to compute, called repeatedly with the same arguments: 

In [ ]:
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)


print(fibonacci(28))

This works, but it's wastefully slow — computing `fibonacci(28)` recomputes `fibonacci(26)` many times over, and `fibonacci(24)` far more times than that, because nothing remembers a result once it's been calculated. `functools.lru_cache` is a decorator that fixes exactly this shape of problem, with one line: 

In [ ]:
from functools import lru_cache


@lru_cache
def fibonacci_cached(n):
    if n < 2:
        return n
    return fibonacci_cached(n - 1) + fibonacci_cached(n - 2)


print(fibonacci_cached(28))

`@lru_cache` wraps `fibonacci_cached` so that every call's arguments and result are remembered; if the same arguments show up again, the cached result is returned immediately, without re-running the function body at all. "LRU" stands for *least recently used* — the cache has a default size limit, and when it's full, the result that hasn't been asked for in the longest time is the one discarded to make room, which keeps the cache from growing without bound across a very long-running program.

This is worth using specifically when a function is **pure** in a particular sense: given the same arguments, it always produces the same result, with no other effects on the rest of the program. Caching a function that reads a changing file, or depends on the current time, would silently return stale answers — the cache has no way to know the "right" answer for a given input just changed.

### `partial` — fixing some arguments of a function ahead of time

Sometimes you have a general-purpose function, and want a more specific version of it with certain arguments already locked in.

In [ ]:
from functools import partial


def power(base, exponent):
    return base ** exponent


square = partial(power, exponent=2)
cube = partial(power, exponent=3)

print(square(5))
print(cube(5))

`partial(power, exponent=2)` returns a new callable that behaves like `power`, except `exponent` is already filled in as `2` — calling `square(5)` is equivalent to calling `power(5, exponent=2)`. This solves a similar problem to what Chapter 47's closures solved by hand (`make_multiplier`, in that chapter's practice problem, did essentially the same thing manually) — `partial` is a ready-made, general-purpose tool for the specific, common case of "the same function, with some arguments already decided."

### The theme across all three tools

`wraps`, `lru_cache`, and `partial` are all, in their own way, solutions to problems that arise directly from treating functions as ordinary values — something Chapter 22 first established and this entire Part III has been building on. `wraps` repairs identity lost when one function wraps another; `lru_cache` avoids redundant work by remembering past calls; `partial` builds a new function out of an old one with some inputs fixed. None of them do anything you couldn't write yourself with a closure — they exist because these particular patterns come up often enough to deserve a tested, standard, one-line solution rather than everyone reimplementing them slightly differently.

### One problem

> Write a function `is_prime(n)` that checks primality by trial division, decorate it with `@lru_cache`, and use `functools.partial` to create a function `is_prime_under_100` that's equivalent to calling `is_prime` — this one is mostly about practicing the imports and syntax correctly, since `partial` here won't change behavior much; the point is seeing both tools used together.

In [ ]:
from functools import lru_cache, partial

# TODO: define is_prime with @lru_cache, then create a partial version


# Part IV — Resource Management and Exceptions

## 50. Context Managers

You've written this pattern many times already, since Basics: 

In [ ]:
with open("notes.txt", "w") as file:
    file.write("Some text\n")

print("file is closed now, even though we never called .close()")

Basics taught you *that* `with` guarantees the file gets closed. This chapter explains *how*, and — more usefully — shows you how to give the same guarantee to resources you define yourself, not just files.

### What `with` is actually doing

`with EXPR as name:` performs a small, fixed sequence of steps, structurally similar to the `iter`/`next` protocol from Chapter 33: it calls a specific method on `EXPR` to set things up, runs the indented block, and then calls another specific method on `EXPR` to tear things down — and critically, that teardown step happens *no matter how* the block finishes, including if an exception is raised partway through.

```text
with obj as name:
    ...
        ↓
obj.__enter__()  is called  →  its return value becomes 'name'
        ↓
the indented block runs
        ↓
obj.__exit__(...)  is called  →  guaranteed, even if the block raised an exception
```

`__enter__` and `__exit__` are the two dunder methods that make an object usable with `with` — a **context manager** is simply any object implementing both. `open(...)` returns exactly such an object: its `__enter__` sets the file up and returns itself (which is why `as file` gives you the file object, ready to use); its `__exit__` closes the file, and it runs that closing code regardless of whether the block finished normally or crashed.

### Writing one from scratch

Here's a minimal context manager, built specifically to make the two steps visible: 

In [ ]:
class Announce:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        print(f"Entering {self.name}")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print(f"Leaving {self.name}")


with Announce("first block") as a:
    print("doing work inside the block")

`"Entering first block"` prints before the block runs, `"doing work inside the block"` during it, `"Leaving first block"` after — in that exact order, because that's exactly the sequence `with` performs. Now the part that actually justifies this chapter's placement right before the exceptions chapter: 

In [ ]:
with Announce("risky block") as a:
    print("about to fail")
    raise ValueError("something went wrong")

`"Leaving risky block"` still printed, immediately, right before the `ValueError` propagated up and crashed the cell. `__exit__` ran despite the exception — that's the entire guarantee `with` provides, and it's exactly why file handling, database connections, and locks are conventionally managed through context managers: whatever cleanup needs to happen (closing a file, releasing a lock, rolling back a transaction) happens reliably, even when something inside the block goes wrong, without you needing a `try`/`finally` written out by hand every single time.

### What those three parameters on `__exit__` are for

`__exit__(self, exc_type, exc_value, traceback)` receives information about whatever exception occurred inside the block — all three arguments are `None` if the block finished without error, and filled in with the exception's details otherwise. This gives `__exit__` the ability to inspect, log, or even deliberately *suppress* an exception (by returning `True` — a capability worth knowing exists, though using it carelessly can hide real bugs, so it's not something to reach for by default).

In [ ]:
class SuppressValueError:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if exc_type is ValueError:
            print(f"Suppressed a ValueError: {exc_value}")
            return True   # tells Python the exception is handled -- don't propagate it
        return False      # anything else, let it propagate normally


with SuppressValueError():
    print("about to fail")
    raise ValueError("this will be caught by __exit__, not by the cell")

print("execution continues normally after the with block")

### One problem

> Write a context manager class `Timer` that prints how long the `with` block took to execute, using `__enter__` to record a start time and `__exit__` to compute and print the elapsed time.

In [ ]:
import time

# TODO: define Timer as a context manager class


## 51. `contextlib.contextmanager`

The `Timer` class from the last chapter's problem, written out in full, needs a class, an `__init__` (if it takes arguments), an `__enter__`, and an `__exit__` — a fair amount of structure for something conceptually simple: "do this before, run the block, do that after." `contextlib.contextmanager` lets you write the exact same behavior as a single generator function.

In [ ]:
from contextlib import contextmanager
import time


@contextmanager
def timer():
    start = time.perf_counter()
    yield
    end = time.perf_counter()
    print(f"Took {end - start:.4f} seconds")


with timer():
    total = sum(range(1_000_000))

print(total)

### How `yield` splits this into "before" and "after"

This connects directly back to Chapter 45's generators, applied in a new way. `@contextmanager` takes a generator function and turns it into something usable with `with`, by treating everything **before** the `yield` as the `__enter__` behavior, and everything **after** the `yield` as the `__exit__` behavior. The `with` block's body runs exactly while the generator is paused at the `yield`.

```text
def timer():
    start = ...        ← runs when the `with` block is entered
    yield               ← the with-block's body runs while paused here
    end = ...           ← runs when the with-block finishes
    print(...)
```

If the generator function's `yield` produces a value, that value is what `as name` captures, exactly matching what `__enter__`'s return value did in the previous chapter: 

In [ ]:
@contextmanager
def announce(name):
    print(f"Entering {name}")
    yield name
    print(f"Leaving {name}")


with announce("second block") as captured_name:
    print("captured:", captured_name)

### Handling an exception inside the block

To react to something going wrong inside the `with` block, wrap the `yield` in a `try`/`finally` (or `try`/`except`), exactly as you would anywhere else — the generator receives control back at the `yield` line specifically when the block raises, the same resumption mechanism from Chapter 45, just triggered by an exception this time rather than a normal `next()` call.

In [ ]:
@contextmanager
def timer():
    start = time.perf_counter()
    try:
        yield
    finally:
        end = time.perf_counter()
        print(f"Took {end - start:.4f} seconds (even though something failed)")


with timer():
    raise ValueError("oops")

The `finally` guarantees the timing message prints even though the block raised — the same guarantee Chapter 50's `__exit__` had automatically, now made explicit through ordinary exception-handling syntax you already know from Basics, rather than through a special method Python calls for you.

### Choosing between the two styles

A class-based context manager (Chapter 50) is the more explicit, more flexible option — particularly once you need to suppress specific exceptions by returning `True` from `__exit__`, or the context manager needs to hold and expose meaningful state through multiple methods. `@contextmanager` is the more concise option, and it's usually the better fit exactly when the pattern is genuinely "run this before, run that after" with nothing more elaborate going on — which describes a large fraction of real context managers.

### One problem

> Rewrite the `Timer` class from the previous chapter's problem as a generator-based context manager using `@contextmanager`.

In [ ]:
from contextlib import contextmanager
import time

# TODO: rewrite Timer using @contextmanager


## 52. Custom Exceptions and Exception Chaining

Basics taught `try`/`except` against exceptions Python already provides — `ValueError`, `ZeroDivisionError`, and so on. Real programs frequently need to signal failures specific to their own domain, in a way generic built-in exceptions can't express clearly.

In [ ]:
def withdraw(balance, amount):
    if amount > balance:
        raise ValueError("Not enough money")
    return balance - amount


withdraw(100, 500)

This works, but `ValueError` is a generic label — it's the same exception type Python raises for a badly formatted number, an invalid enum value, or dozens of unrelated situations. Code that wants to specifically catch "insufficient funds," and nothing else, has no reliable way to distinguish this `ValueError` from any other.

### Defining your own exception type

An exception is just a class — specifically, one that inherits (directly or indirectly) from Python's built-in `Exception`.

In [ ]:
class InsufficientFundsError(Exception):
    pass


def withdraw(balance, amount):
    if amount > balance:
        raise InsufficientFundsError("Not enough money")
    return balance - amount


try:
    withdraw(100, 500)
except InsufficientFundsError as e:
    print("Caught it specifically:", e)

Now `except InsufficientFundsError:` catches exactly this situation and nothing else — a genuine, unrelated `ValueError` elsewhere in the same `try` block would not be caught by it, which is precisely the specificity a generic `ValueError` couldn't offer.

### Building a small hierarchy

Because a custom exception is a class, it can participate in inheritance exactly like any other class from Part II — letting you catch either a broad category or one specific member of it, depending on what the calling code actually needs.

In [ ]:
class BankError(Exception):
    pass


class InsufficientFundsError(BankError):
    pass


class InvalidAmountError(BankError):
    pass


def withdraw(balance, amount):
    if amount <= 0:
        raise InvalidAmountError("Amount must be positive")
    if amount > balance:
        raise InsufficientFundsError("Not enough money")
    return balance - amount


for amount in [-10, 500, 50]:
    try:
        print(withdraw(100, amount))
    except InsufficientFundsError:
        print("Specifically: not enough funds")
    except InvalidAmountError:
        print("Specifically: bad amount")
    except BankError:
        print("Some other bank-related problem")

`BankError` here is never raised directly — it exists as a shared base, letting calling code choose its own level of precision: catch `InsufficientFundsError` specifically if that's the only case needing special handling, or catch `BankError` broadly if the goal is just "something bank-related went wrong, log it and move on," without needing to enumerate every specific subclass by hand.

### Why swallowing every exception silently is a real anti-pattern

It might look tempting to write `except Exception:` (or worse, a bare `except:`) everywhere, to guarantee nothing ever crashes. This is worth actively discouraging: it catches genuine programming bugs — a typo'd variable name causing a `NameError`, an actual logic error — exactly as readily as it catches the specific, anticipated failure you meant to handle, and hides all of them equally, silently. A program that "never crashes" because every error is swallowed indiscriminately isn't more reliable; it's simply failing quietly, somewhere you can no longer see, which is generally worse than failing loudly at the point something actually went wrong.

### `raise ... from ...` — chaining exceptions deliberately

Sometimes handling one exception requires raising a different, more meaningful one in its place — while still preserving the original as context, for debugging.

In [ ]:
def load_config(text):
    try:
        amount = float(text)
    except ValueError as original_error:
        raise InvalidAmountError(f"Could not parse amount: {text!r}") from original_error
    return amount


load_config("not-a-number")

Notice the traceback this produces: it shows *both* exceptions — the original `ValueError`, explicitly labeled as "the direct cause of the following exception," followed by the `InvalidAmountError` that was deliberately raised in its place. `from original_error` is what creates this link. Without it, Python still shows both if a new exception happens to be raised while handling another (labeled "during handling of the above exception"), but `from` makes that relationship explicit and intentional, telling anyone reading the traceback "this second failure is a direct, deliberate consequence of the first one, not an unrelated accident that happened to occur during cleanup."

### One problem

> Define an exception hierarchy for a simple file-processing tool: a base `FileProcessingError`, with subclasses `UnsupportedFormatError` and `EmptyFileError`. Write a function `process_file(filename, contents)` that raises the appropriate one, and demonstrate catching both specifically and via the shared base.

In [ ]:
# TODO: define the exception hierarchy and process_file as described


# Part V — Working With Real Data

## 53. Regular Expressions with `re`

Checking whether a string looks like a valid pattern using only what Basics taught quickly becomes unwieldy: 

In [ ]:
def looks_like_phone_number(text):
    digits = text.replace("-", "")
    return len(digits) == 10 and digits.isdigit()


print(looks_like_phone_number("555-123-4567"))
print(looks_like_phone_number("hello"))

This works for exactly one format, and any variation — different separators, an optional area code in parentheses — needs its own hand-written special case. A **regular expression** (regex) is a compact, standardized language for describing patterns in text, letting you express "ten digits, optionally separated by dashes" directly, rather than writing custom logic for every variation.

In [ ]:
import re

pattern = r"^\d{3}-\d{3}-\d{4}$"

print(re.search(pattern, "555-123-4567"))
print(re.search(pattern, "5551234567"))

### Raw strings, and why they matter here specifically

`r"^\d{3}-\d{3}-\d{4}$"` uses a **raw string** — the `r` prefix from Chapter 8's slicing examples, repurposed here for a genuinely important reason. Regular expressions use backslash sequences (`\d`, `\w`, `\s`, and others) that mean something specific to the *regex engine*, but an ordinary Python string also treats backslashes specially (`\n` for a newline, for instance), and the two sets of meanings can collide. A raw string tells Python "don't interpret backslashes at all here — pass them through exactly as typed," so `\d` reaches the regex engine as the two characters `\` and `d`, meaning "any digit," rather than Python trying (and generally failing) to interpret `\d` as some escape sequence of its own. Writing regex patterns as raw strings is close to a universal convention, worth adopting from the start.

### The building blocks

```text
.       any character (except a newline)
\d      any digit
\w      any "word" character -- letter, digit, or underscore
\s      any whitespace character
^       the start of the string
$       the end of the string
{n}     exactly n repetitions of whatever came before it
+       one or more repetitions
*       zero or more repetitions
?       zero or one repetition (makes something optional)
```

`^\d{3}-\d{3}-\d{4}$` reads as: start of string, exactly three digits, a dash, exactly three digits, a dash, exactly four digits, end of string. The `^` and `$` anchors matter — without them, the pattern would match *anywhere inside* a longer string, not just when the whole string fits the shape.

In [ ]:
loose_pattern = r"\d{3}-\d{3}-\d{4}"
strict_pattern = r"^\d{3}-\d{3}-\d{4}$"

text = "call 555-123-4567 today"

print(re.search(loose_pattern, text))    # finds it embedded in the sentence
print(re.search(strict_pattern, text))   # None -- the whole string isn't just the number

### `search`, `match`, `findall`, and `sub`

`re.search` looks for the pattern anywhere in the string and returns a **match object** describing the first occurrence, or `None`. `re.match` is similar but only checks at the very start of the string. `re.findall` returns every non-overlapping match, as a list. `re.sub` replaces matches with something else.

In [ ]:
text = "Contact us at 555-123-4567 or 555-987-6543"

numbers = re.findall(r"\d{3}-\d{3}-\d{4}", text)
print(numbers)

masked = re.sub(r"\d{3}-\d{3}-\d{4}", "[phone number removed]", text)
print(masked)

### Groups — pulling a pattern apart into named pieces

Parentheses in a pattern mark a **group**, letting you extract a specific portion of what matched, rather than only the whole thing.

In [ ]:
match = re.search(r"(\d{3})-(\d{3})-(\d{4})", "555-123-4567")

print(match.group(0))   # the whole match
print(match.group(1))   # the first group -- area code
print(match.groups())   # all groups, as a tuple

Groups can also be given names, which makes the extracted data self-documenting rather than relying on remembering "group 1 is the area code": 

In [ ]:
match = re.search(r"(?P<area>\d{3})-(?P<exchange>\d{3})-(?P<line>\d{4})", "555-123-4567")
print(match.group("area"))
print(match.groupdict())

### A note on when *not* to reach for regex

Regular expressions are the right tool for genuine pattern matching — validating a shape, extracting a substructure, replacing all occurrences of a pattern. They're frequently the *wrong* tool for structured formats that already have a proper parser — trying to extract data from JSON or HTML with regex is a well-known way to create fragile, hard-to-maintain code, when a dedicated tool (JSON parsing is the next chapter's neighbor, Chapter 55) already exists and handles the format's real structure correctly.

### One problem

> Write a function `extract_emails(text)` that uses `re.findall` to return every email address in a block of text (a reasonable simplified pattern is fine — something like one-or-more word characters or dots, an `@`, then a domain).

In [ ]:
import re

# TODO: define extract_emails(text)


## 54. Dates and Times

Representing a date as separate numbers works, but poorly: 

In [ ]:
year, month, day = 2026, 9, 3

# What day of the week is this? How many days until another date?
# Nothing here helps answer either question directly.
print(year, month, day)

The `datetime` module provides proper types for exactly this kind of data, along with the arithmetic and formatting operations that plain numbers don't give you for free.

In [ ]:
from datetime import date, time, datetime

d = date(2026, 9, 3)
t = time(14, 30)
dt = datetime(2026, 9, 3, 14, 30)

print(d)
print(t)
print(dt)

`date` holds a calendar date with no time component; `time` holds a time of day with no date attached; `datetime` combines both. Each is an object with meaningful attributes and methods, not just a formatted string: 

In [ ]:
print(dt.year, dt.month, dt.day)
print(dt.weekday())   # Monday is 0, Sunday is 6

### `timedelta` — the difference between two points in time

Subtracting one date from another produces a `timedelta`, representing a *span* of time rather than a point in time — a genuinely different kind of value, the same way `int` and `float` were genuinely different kinds of number back in Basics.

In [ ]:
from datetime import date, timedelta

start = date(2026, 1, 1)
end = date(2026, 9, 3)

difference = end - start
print(difference)
print(difference.days)

A `timedelta` can also be added to a date or datetime, to compute a point in time relative to a known one — a natural, readable alternative to manually tracking days-per-month and leap years yourself: 

In [ ]:
today = date(2026, 9, 3)
next_week = today + timedelta(days=7)
print(next_week)

ninety_days_ago = today - timedelta(days=90)
print(ninety_days_ago)

### Formatting and parsing

Converting a `datetime` into a specific text format, or the reverse, uses a small pattern language of its own — `strftime` ("string format time") to produce text, `strptime` ("string parse time") to read it back.

In [ ]:
dt = datetime(2026, 9, 3, 14, 30)

print(dt.strftime("%Y-%m-%d"))
print(dt.strftime("%B %d, %Y at %I:%M %p"))

In [ ]:
text = "2026-09-03 14:30"
parsed = datetime.strptime(text, "%Y-%m-%d %H:%M")
print(parsed)
print(type(parsed))

The format codes (`%Y` for a four-digit year, `%m` for a zero-padded month, `%d` for a zero-padded day, and so on) have to match the actual shape of the text exactly, or `strptime` raises a `ValueError` — worth wrapping in a `try`/`except` (Chapter 25) when parsing text you don't fully control, such as user input.

### A brief, honest word on timezones

Every `datetime` object shown so far is what's called **naive** — it has no attached information about which timezone it belongs to, and Python will not stop you from comparing or subtracting naive datetimes that silently assume different, unstated timezones. An **aware** datetime carries explicit timezone information, using a `tzinfo` object: 

In [ ]:
from datetime import timezone

aware_dt = datetime(2026, 9, 3, 14, 30, tzinfo=timezone.utc)
print(aware_dt)
print(aware_dt.tzinfo)

Full, correct timezone handling — daylight saving transitions, historical timezone rule changes, converting between named zones like `"America/New_York"` — is a genuinely deep topic with its own dedicated library (`zoneinfo`, in modern Python) rather than something this chapter can responsibly compress into a paragraph. What's worth taking away now is the distinction itself: know whether the datetimes you're working with are naive or aware, and never assume two naive datetimes from different sources actually refer to the same timezone just because Python lets you compare them without complaint.

### One problem

> Write a function `days_until_birthday(birth_month, birth_day)` that returns how many days remain until that birthday, measured from today's date (use `date.today()`), handling the case where the birthday has already passed this year by calculating for next year instead.

In [ ]:
from datetime import date

# TODO: define days_until_birthday(birth_month, birth_day)


## 55. Serialization — JSON, CSV, and `pickle`

Basics showed you how to write text to a file. A Python dictionary, though, doesn't turn into a file's worth of text on its own: 

In [ ]:
person = {"name": "Ali", "age": 25, "hobbies": ["reading", "chess"]}

with open("person.txt", "w") as file:
    file.write(str(person))

with open("person.txt", "r") as file:
    contents = file.read()

print(contents)
print(type(contents))

`str(person)` produced text that *looks* like the dictionary, but it's just a string — there's no way to get the original dictionary back from it safely and generally (you'd have to write your own parser, which is a bad use of effort for a completely solved problem). **Serialization** is the general term for converting a Python value into a form that can be stored or transmitted, and reliably converted back later. JSON is the most common serialization format for exactly this kind of structured data.

### JSON

JSON (JavaScript Object Notation) looks almost like Python's own dictionary and list syntax, which is exactly why it maps so cleanly to and from Python's built-in types.

In [ ]:
import json

person = {"name": "Ali", "age": 25, "hobbies": ["reading", "chess"]}

with open("person.json", "w") as file:
    json.dump(person, file)

with open("person.json", "r") as file:
    loaded = json.load(file)

print(loaded)
print(loaded == person)
print(type(loaded))

`json.dump(person, file)` writes `person` to the file in JSON's text format; `json.load(file)` reads it back and reconstructs an actual Python dictionary — not merely text that resembles one. `loaded == person` being `True` confirms the round trip preserved the data faithfully.

`json.dumps` and `json.loads` (note the trailing `s`, for "string") do the same conversion without touching a file at all, useful whenever the JSON text itself needs to be sent somewhere rather than saved — over a network request, for instance.

In [ ]:
text = json.dumps(person)
print(text)
print(type(text))

back = json.loads(text)
print(back)

### What JSON can and can't represent

JSON only understands a small, fixed set of types: strings, numbers, booleans, `null` (Python's `None`), lists, and objects (Python's dictionaries, with string keys). A Python-specific type — a `datetime`, a custom class instance, a set — has no direct JSON equivalent, and `json.dump` raises an error if you try: 

In [ ]:
from datetime import date

data = {"today": date.today()}
json.dumps(data)

`TypeError: Object of type date is not JSON serializable`. Fixing this means converting the problematic value to something JSON *does* understand first — a string, typically, using exactly the `strftime` tool from the last chapter — rather than expecting `json` to guess how you want it represented.

### CSV — tabular data, row by row

CSV (comma-separated values) is the standard format for simple, spreadsheet-shaped data — rows and columns, rather than JSON's nested structure.

In [ ]:
import csv

rows = [
    ["name", "age", "city"],
    ["Ali", "25", "Lahore"],
    ["Sara", "30", "Karachi"],
]

with open("people.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerows(rows)

with open("people.csv", "r", newline="") as file:
    reader = csv.reader(file)
    for row in reader:
        print(row)

`csv.writer` and `csv.reader` handle the fiddly details of the format correctly — quoting a value that itself contains a comma, for instance — which is why using the module is worth doing even for a format simple enough that you could, in principle, build the text yourself with string joins. `DictReader` and `DictWriter` are worth knowing exist too, for exactly the common case where the first row is a header naming each column: 

In [ ]:
with open("people.csv", "r", newline="") as file:
    reader = csv.DictReader(file)
    for row in reader:
        print(row["name"], "is", row["age"], "years old")

### `pickle` — serializing arbitrary Python objects, with a real warning attached

Unlike JSON and CSV, `pickle` can serialize almost any Python object — custom class instances included — because it's specific to Python itself rather than being a cross-language, human-readable format.

In [ ]:
import pickle

person = {"name": "Ali", "hobbies": ["reading", "chess"]}

with open("person.pkl", "wb") as file:   # note "wb" -- pickle produces binary data
    pickle.dump(person, file)

with open("person.pkl", "rb") as file:
    loaded = pickle.load(file)

print(loaded)

### The warning this chapter promised

**Never call `pickle.load` on data from a source you don't fully trust.** Unlike JSON, which can only ever produce inert data (strings, numbers, lists, dictionaries), unpickling data is capable of executing arbitrary code as a side effect of the loading process itself — a maliciously constructed pickle file can run harmful code the instant it's loaded, before your program ever does anything with the result. This isn't a hypothetical edge case; it's a well-documented, actively exploited category of vulnerability. Use `pickle` freely for your own program's own trusted data (caching an intermediate result to reload later, for instance) — never for data received from a user, a network request, or any other source you don't fully control.

### One problem

> Given a list of dictionaries representing products (`name`, `price`, `in_stock`), write them to both a JSON file and a CSV file, then read each back and confirm the data matches, printing what (if anything) had to change in the CSV version compared to the original Python types.

In [ ]:
import json
import csv

products = [
    {"name": "Widget", "price": 9.99, "in_stock": True},
    {"name": "Gadget", "price": 19.99, "in_stock": False},
]

# TODO: write products to both products.json and products.csv, then read both back


## 56. `map`, `filter`, and Functional-Style Tools

Chapter 31 rewrote a manual loop as a list comprehension. There's an older, related style worth knowing, because you'll encounter it in real code, and because it deepens the "functions are ordinary values" idea from Chapter 47 onward.

In [ ]:
numbers = [1, 2, 3, 4, 5]

squares = list(map(lambda n: n ** 2, numbers))
print(squares)

### `lambda` — a function without a name

`lambda n: n ** 2` defines a small, anonymous function — equivalent to: 

In [ ]:
def square(n):
    return n ** 2

print(square(5))

square_lambda = lambda n: n ** 2
print(square_lambda(5))

A `lambda` can only contain a single expression (no statements, no multiple lines), and that expression's value is automatically returned — there's no `return` keyword inside one. It exists for exactly the situation above: a function that's only needed once, briefly, as an argument to something else, where writing out a full `def` elsewhere in the file and naming it would add ceremony without adding clarity.

### `map` — applying a function to every item

`map(function, iterable)` applies `function` to each item of `iterable`, producing the results one at a time. Notice `map` itself returned something that needed `list(...)` to actually see: 

In [ ]:
numbers = [1, 2, 3, 4, 5]

mapped = map(lambda n: n ** 2, numbers)
print(mapped)
print(list(mapped))

`map` is **lazy**, in exactly the sense Chapter 46 used that word for generator expressions — it doesn't compute any squared value until something actually asks for one, by iterating over it. This isn't a coincidence: `map` genuinely is implemented as something very close to a generator internally, applying the same lazy-evaluation idea to "transform every item," rather than "produce every item of a range."

### `filter` — keeping only what matches

`filter(function, iterable)` keeps only the items where `function` returns something truthy, discarding the rest — following the same truthiness rules from Chapter 11.

In [ ]:
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

evens = list(filter(lambda n: n % 2 == 0, numbers))
print(evens)

`filter` is lazy in the same way `map` is — nothing is checked until iterated over.

### Side by side with the comprehension equivalent

It's worth putting both styles next to each other directly, because this is genuinely a matter of readability and taste rather than one being objectively superior: 

In [ ]:
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# map/filter style
result_a = list(map(lambda n: n ** 2, filter(lambda n: n % 2 == 0, numbers)))

# comprehension style
result_b = [n ** 2 for n in numbers if n % 2 == 0]

print(result_a)
print(result_b)
print(result_a == result_b)

Most Python style guidance, and most Python code you'll encounter in practice, leans toward the comprehension version for exactly this kind of case — `[n ** 2 for n in numbers if n % 2 == 0]` reads left to right as one coherent sentence, where the `map`/`filter` version has to be read from the inside outward (`filter` happens first, then `map` wraps around it), and stacking more than one `map`/`filter` call quickly becomes harder to follow than the equivalent comprehension.

### Where `map` and `filter` genuinely earn their place

They're a better fit than a comprehension when the function being applied already exists, has a real name, and doesn't need a `lambda` wrapped around it at all: 

In [ ]:
words = ["123", "456", "789"]

numbers = list(map(int, words))
print(numbers)

`map(int, words)` reads cleanly as "convert every word to an int" — there's no `lambda` needed, because `int` itself is already a callable that takes one argument and returns a value, exactly what `map` requires. The equivalent comprehension, `[int(w) for w in words]`, is perfectly reasonable too; this is genuinely one of the places where either style is a fine, common choice, and reaching for whichever one reads more naturally for the specific case at hand is a reasonable way to decide, rather than adopting an absolute rule for every situation.

### One problem

> Given a list of strings representing prices like `["19.99", "5.50", "100.00"]`, use `map` to convert them to floats, and separately use `filter` to keep only the ones above `10.00`, then print both results.

In [ ]:
prices_text = ["19.99", "5.50", "100.00"]

# TODO: use map to convert to floats, and filter to keep those above 10.00


# Part VI — Better Data Structures

## 57. The `collections` Module

Each tool in this chapter starts from a real, slightly annoying problem you already have the tools to solve manually — worth doing once by hand, to see exactly what the specialized tool is saving you from.

### Counting things — `Counter`

Counting occurrences with a plain dictionary means handling the "have I seen this key before" case every time: 

In [ ]:
words = ["apple", "banana", "apple", "cherry", "banana", "apple"]

counts = {}
for word in words:
    counts[word] = counts.get(word, 0) + 1

print(counts)

In [ ]:
from collections import Counter

counts = Counter(words)
print(counts)
print(counts.most_common(2))

`Counter(words)` does exactly the loop above in one call, and adds useful extras on top — `.most_common(n)` returns the `n` most frequent items, sorted, which you'd otherwise have to build yourself with `sorted()` and a key function.

### A dictionary with a built-in default — `defaultdict`

The `counts.get(word, 0)` pattern above — "look this up, and if it's missing, use a sensible default instead" — comes up constantly beyond just counting, particularly when the default is itself a mutable collection.

In [ ]:
groups = {}
words = ["apple", "avocado", "banana", "blueberry", "cherry"]

for word in words:
    first_letter = word[0]
    if first_letter not in groups:
        groups[first_letter] = []
    groups[first_letter].append(word)

print(groups)

In [ ]:
from collections import defaultdict

groups = defaultdict(list)
for word in words:
    groups[word[0]].append(word)

print(dict(groups))

`defaultdict(list)` creates a dictionary that, the moment you access a key it doesn't yet have, automatically creates that key with an empty list (because `list` — the type itself, not a call to it — was passed in as the "default factory") before returning it. `groups[word[0]].append(word)` never needs to check whether the key already exists; if it doesn't, `defaultdict` silently creates it with a fresh empty list first. This removes exactly the `if key not in groups:` check the manual version needed, and generalizes to any default value you can build with a zero-argument callable — `defaultdict(int)` for counting (giving `0` as the default, which is what `Counter` does more conveniently), `defaultdict(set)` for grouping unique values, and so on.

### A double-ended queue — `deque`

Removing from the front of a plain list is a real, easy-to-miss performance trap: every removal from the front has to shift every remaining element over by one position, which becomes noticeably slow as a list grows large. A `deque` (pronounced "deck," short for double-ended queue) is built specifically to make additions and removals from *either* end efficient.

In [ ]:
from collections import deque

queue = deque(["first", "second", "third"])

queue.append("fourth")        # add to the right end
queue.appendleft("zeroth")    # add to the left end

print(queue)

print(queue.popleft())        # remove and return from the left end
print(queue)

`deque` supports everything a list does for ordinary indexing and iteration, while adding `appendleft` and `popleft` as genuinely efficient operations — the right structure whenever something is naturally processed in strict first-in-first-out order (a real, waiting-line-style queue) rather than accessed by arbitrary position.

### A lightweight, named alternative to a plain tuple — `namedtuple`

Chapter 18 showed unpacking a `point = (3, 4)` tuple into `x, y`. That works, but reading `point[0]` and `point[1]` elsewhere in a longer program tells you nothing about *which* coordinate is which without checking back to where the tuple was created.

In [ ]:
from collections import namedtuple

Point = namedtuple("Point", ["x", "y"])

p = Point(3, 4)
print(p.x, p.y)
print(p[0], p[1])   # still works positionally too
print(p)

`namedtuple("Point", ["x", "y"])` creates a new, lightweight class — `Point` — whose instances behave exactly like ordinary tuples (immutable, indexable, unpackable) while *also* supporting attribute access by name, `p.x` and `p.y`, which is far more self-documenting than remembering that position `0` means the x-coordinate. This is worth remembering as a genuine stepping stone toward Chapter 58's dataclasses, which solve a closely related problem with a somewhat different, more flexible set of trade-offs.

### One problem

> Given a list of words, use `Counter` to find the three most common word lengths (not the words themselves — the *lengths* of the words), and use a `defaultdict(list)` to group the original words by their length.

In [ ]:
from collections import Counter, defaultdict

words = ["a", "an", "the", "cat", "dog", "elephant", "it", "hi", "sun", "run"]

# TODO: find the three most common lengths, and group words by length


## 58. Dataclasses

A class whose entire job is holding a handful of related values ends up writing a fair amount of boilerplate for very little unique logic: 

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Point(x={self.x!r}, y={self.y!r})"

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y


p1 = Point(1, 2)
p2 = Point(1, 2)
print(p1)
print(p1 == p2)

None of this is difficult, individually — `__init__` assigning parameters to attributes, a `__repr__` for readable printing (Chapter 42), an `__eq__` comparing all the fields — but it's exactly the same shape every time you write a class like this, for every new set of fields, and it's easy to forget one piece (an `__eq__`, say) and only notice much later when comparing two instances silently does the wrong thing.

### `@dataclass` — the same class, generated for you

A `@dataclass` decorator, applied to a class that only declares its fields with type hints, generates `__init__`, `__repr__`, and `__eq__` automatically.

In [ ]:
from dataclasses import dataclass


@dataclass
class Point:
    x: int
    y: int


p1 = Point(1, 2)
p2 = Point(1, 2)

print(p1)
print(p1 == p2)

Compare this directly against the manual version above: identical behavior, a fraction of the code. `x: int` and `y: int` here aren't merely documentation the way a bare type hint sometimes is (Chapter 29's warning that hints aren't enforced still applies) — `@dataclass` actually *reads* these annotations to know what fields to generate `__init__` and the rest around, which is a genuinely new, functional use for type hints beyond what Basics introduced them for.

### Fields with default values

Exactly like an ordinary function parameter, a field can specify a default: 

In [ ]:
@dataclass
class Point:
    x: int
    y: int
    label: str = "unnamed"


p1 = Point(1, 2)
p2 = Point(3, 4, label="origin-adjacent")

print(p1)
print(p2)

One genuine trap worth knowing about directly, because it echoes Chapter 36's mutable class attribute warning almost exactly: a field cannot default to a plain mutable value like `[]` or `{}`.

In [ ]:
@dataclass
class Basket:
    items: list = []

`ValueError: mutable default <class 'list'> for field items is not allowed`. This isn't `@dataclass` being needlessly strict — it's protecting you from exactly the shared-mutable-default bug Chapter 36 walked through in detail, where every instance would otherwise end up silently sharing the same one list. The correct way to give a field a fresh, independent mutable default uses `field(default_factory=...)`: 

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Basket:
    items: list = field(default_factory=list)


b1 = Basket()
b2 = Basket()

b1.items.append("apple")

print(b1.items)
print(b2.items)

`default_factory=list` tells the generated `__init__` to call `list()` fresh, separately, for every new instance — producing a genuinely independent empty list each time, rather than one shared list reused everywhere, exactly the fix Chapter 36 applied by hand inside a manual `__init__`.

### Frozen dataclasses — opting into immutability

Passing `frozen=True` makes every instance immutable after creation, the same guarantee tuples and strings have always had.

In [ ]:
@dataclass(frozen=True)
class Point:
    x: int
    y: int


p = Point(1, 2)
p.x = 100

`FrozenInstanceError` — a deliberate design choice, worth reaching for whenever a value is meant to represent something fixed (a coordinate, a configuration setting, a completed record) that should never be quietly mutated somewhere later in a program, in the same spirit as Chapter 18's discussion of why tuples exist alongside lists.

### When a dataclass is, and isn't, the right choice

A dataclass is an excellent fit for what this chapter's examples show: a small bundle of related values, with mostly-generated behavior. It becomes the wrong tool the moment a class's real purpose is substantial custom behavior — several nontrivial methods, careful encapsulation through properties, a deep inheritance hierarchy — where the auto-generated `__init__`/`__repr__`/`__eq__` are only a small, incidental part of what the class actually does. Nothing stops a dataclass from having ordinary methods too, so the two styles aren't mutually exclusive, but if you find yourself overriding most of what `@dataclass` generates for you, that's a sign a plain class might be the more honest fit.

### One problem

> Define a `@dataclass` named `Book` with `title: str`, `author: str`, `pages: int`, and `tags: list` (defaulting to an independent empty list per instance). Create two books, add different tags to each, and confirm they don't share the same list.

In [ ]:
from dataclasses import dataclass, field

# TODO: define Book as a dataclass with a correct list default


# Part VII — Type Thinking

## 59. Type Hints Beyond the Basics

Basics ended with `age: int` and `def add(a: int, b: int) -> int:` — hints for the simple case where a value is always exactly one type. Real functions frequently need to express something more nuanced than that, and Python's typing tools have grown to match.

### `Optional` — a value that might be `None`

A function that sometimes has nothing to return often uses `None` to signal that.

In [ ]:
from typing import Optional


def find_user(user_id: int, users: dict) -> Optional[str]:
    return users.get(user_id)


users = {1: "Ali", 2: "Sara"}
print(find_user(1, users))
print(find_user(99, users))

`Optional[str]` means "either a `str`, or `None`" — a more honest hint than plain `str` for a function whose real behavior includes the possibility of returning nothing. This connects directly to Chapter 19's `.get()`, which returns `None` for a missing key precisely — the hint is documenting a behavior the function already legitimately has, not introducing new behavior.

### `Union` — a value that could be more than one specific type

`Optional[X]` is actually shorthand for a more general tool, `Union`, which lets you name any set of acceptable types.

In [ ]:
from typing import Union


def normalize_id(value: Union[int, str]) -> str:
    return str(value)


print(normalize_id(42))
print(normalize_id("42"))

`Union[int, str]` means "an `int` or a `str`, either is acceptable here" — and `Optional[str]` from a moment ago is exactly `Union[str, None]`, spelled more conveniently for that one extremely common case. Python 3.10 and later also allows a shorter syntax for the same idea, using `|` directly between types: 

In [ ]:
def normalize_id(value: int | str) -> str:
    return str(value)

print(normalize_id(42))

Both spellings mean the same thing; `|` is simply newer and more concise, and you'll see both in real code depending on how recently it was written.

### `Literal` — not just a type, but specific allowed values

Sometimes a parameter isn't just "any string" — it's one of a small, fixed set of specific strings, and only those.

In [ ]:
from typing import Literal


def set_status(status: Literal["active", "inactive", "pending"]) -> None:
    print(f"Status set to {status}")


set_status("active")
set_status("cancelled")   # a type checker would flag this -- not one of the allowed values

`set_status("cancelled")` still *runs* — exactly as Chapter 29 emphasized, Python itself never enforces hints at runtime — but a type checker, reading this code without running it at all, can now flag that call as wrong, because `"cancelled"` isn't among the `Literal` values declared. This is a case where the hint captures something a plain `str` genuinely could not: not just the type, but the specific legal values.

### Hinting the contents of a collection, more precisely

Basics briefly showed `list[float]`. That generalizes naturally to dictionaries and other containers, including nested combinations: 

In [ ]:
def average_by_category(data: dict[str, list[float]]) -> dict[str, float]:
    return {category: sum(values) / len(values) for category, values in data.items()}


scores = {"math": [90.0, 85.0], "science": [78.0, 92.0, 88.0]}
print(average_by_category(scores))

`dict[str, list[float]]` reads as "a dictionary whose keys are strings and whose values are lists of floats" — precise enough that a reader (or a type checker) knows exactly what shape of data the function expects, without needing to read the function body to guess.

### Type aliases — naming a complicated hint

Once a hint like `dict[str, list[float]]` gets used in more than one place, repeating it is both tedious and a maintenance risk (change the shape in one spot, forget another). A type alias gives it a name.

In [ ]:
ScoresByCategory = dict[str, list[float]]


def average_by_category(data: ScoresByCategory) -> dict[str, float]:
    return {category: sum(values) / len(values) for category, values in data.items()}


def total_by_category(data: ScoresByCategory) -> dict[str, float]:
    return {category: sum(values) for category, values in data.items()}

`ScoresByCategory` is just a regular variable, assigned to a type expression — nothing special is happening syntactically, but by convention it's named in `PascalCase`, matching class-naming conventions, to visually distinguish "this name refers to a type" from an ordinary variable.

### Why this matters more as a codebase grows

None of this changes what Python actually executes — every warning from Chapter 29 about hints being purely informational still holds completely. The value compounds specifically as a codebase gets larger: a function signature that precisely states "an int or a str," "one of these three literal strings," or "a dictionary shaped exactly like this" tells a reader — or an automated type checker run before the code ever executes — far more than a bare, untyped parameter ever could, at the cost of a small amount of extra syntax that never affects runtime behavior at all.

### One problem

> Write a function `parse_setting(value: str) -> Union[int, float, bool]` that attempts to interpret a string as an `int` first, then a `float`, then falls back to treating `"true"`/`"false"` (case-insensitive) as a `bool`, raising `ValueError` if none apply.

In [ ]:
from typing import Union

# TODO: define parse_setting as described


## 60. Protocols and Structural Typing

Chapter 40 introduced duck typing: `announce(shape)` worked on anything with an `.area()` method, with no shared inheritance required. Chapter 41 then introduced the opposite instinct: an abstract base class, formally requiring subclasses to implement specific methods, enforced at the moment an object is created. `Protocol` offers a third option, and it's worth seeing exactly what gap it fills between the other two.

### The gap between duck typing and ABCs

Duck typing is flexible but gives a type checker (and a careful reader) nothing to verify ahead of time — the `announce("just a string")` failure from Chapter 40 wasn't caught until the exact line that failed. An ABC fixes that, but at a real cost: every type that wants to qualify must explicitly inherit from it, which is often impossible or inappropriate for types you don't control — a class from a library you're using, say, that already has a perfectly good `.area()` method but was never written to inherit from your `Shape`.

In [ ]:
from typing import Protocol


class HasArea(Protocol):
    def area(self) -> float:
        ...


def announce(shape: HasArea) -> None:
    print(f"This shape has an area of {shape.area()}")


class Circle:
    def __init__(self, radius: float):
        self.radius = radius

    def area(self) -> float:
        return 3.14159 * self.radius ** 2


announce(Circle(3))

Notice `Circle` never mentions `HasArea` anywhere — no inheritance, no explicit declaration of any relationship at all. This is **structural typing**: `Circle` satisfies `HasArea` purely because it *has the right shape* — a method named `area`, taking no extra arguments, returning something usable as a `float` — not because of any declared family relationship. "Structural" is the precise term for exactly what duck typing was always doing informally; `Protocol` gives that same idea a name a type checker can actually verify ahead of time.

### What a `Protocol` actually specifies

The `...` bodies inside `HasArea` are never meant to run — just as with `@abstractmethod` in Chapter 41, a `Protocol`'s methods only exist to declare a required shape, never to provide real behavior. Unlike an ABC, though, nothing about `Protocol` is checked when an object is actually created; there's no runtime enforcement at all by default. Its entire value lives in what a type checker can verify *before* the code runs: given `announce`'s signature, a type checker can look at any call to `announce(...)` and confirm the argument has a compatible `.area()` method, flagging a mismatch as an error during development, without ever needing that type to inherit from anything.

### Relating this back to duck typing directly

`Protocol` doesn't replace duck typing — it's a way of *describing*, precisely and checkably, exactly the kind of behavior-based compatibility duck typing already relies on at runtime. Where an ABC says "you must formally declare membership in this family before I'll allow you to be used here," a `Protocol` says "I don't care about your family — I only care whether you shape up to look like this," and then lets tooling verify that shape ahead of time instead of leaving it to be discovered by an eventual crash. This is a genuinely close match to Python's own general philosophy from Chapter 40 — judge by behavior, not lineage — now given a form that static tools can reason about.

### One problem

> Define a `Protocol` named `Serializable` requiring a method `to_dict(self) -> dict`. Write two unrelated classes that each implement `to_dict` differently, and a function `save_all(items: list[Serializable])` that converts each item and prints the resulting dictionaries.

In [ ]:
from typing import Protocol

# TODO: define Serializable, two implementing classes, and save_all


# Part VIII — Testing and Project Structure

## 61. Testing — Proving Code Works

Every function in this book so far has been checked the same way: write it, call it once or twice, look at the output, decide it seems right. That approach has a real limit — it verifies the cases you happened to think to try, at the moment you happened to try them, and gives you no way to know, later, whether a change you made somewhere else quietly broke something that used to work.

### The core idea: an assertion

`assert` is a statement that does nothing if its condition is true, and raises an `AssertionError` if it's false.

In [ ]:
def add(a, b):
    return a + b


assert add(2, 3) == 5
print("passed")

In [ ]:
assert add(2, 3) == 6
print("this line never runs")

That's genuinely the entire mechanism testing is built on: state what you expect, and let Python tell you loudly if reality disagrees. The value isn't in the syntax — it's in the discipline of writing these statements down permanently, as actual code, rather than only checking by eye once and trusting your memory afterward.

### From one assertion to a real test function

A **test case** is a function whose entire job is checking one specific piece of expected behavior, using one or more assertions.

In [ ]:
def add(a, b):
    return a + b


def test_add_two_positive_numbers():
    assert add(2, 3) == 5


def test_add_negative_numbers():
    assert add(-2, -3) == -5


def test_add_zero():
    assert add(5, 0) == 5


test_add_two_positive_numbers()
test_add_negative_numbers()
test_add_zero()
print("all tests passed")

Naming each test function specifically (`test_add_zero`, not `test_2`) matters more than it might seem — when a test fails, its *name* is often the first, and sometimes only, description of exactly what expectation broke, especially once there are dozens of them and you can't inspect every assertion by eye.

### Edge cases — testing more than the obvious case

`add(2, 3)` is the case that's easy to think of; it's rarely the case that actually reveals a bug. **Edge cases** — the boundary and unusual situations a function might handle incorrectly — deserve deliberate attention.

In [ ]:
def average(numbers):
    return sum(numbers) / len(numbers)


def test_average_normal_case():
    assert average([1, 2, 3]) == 2


def test_average_single_number():
    assert average([5]) == 5


def test_average_empty_list():
    try:
        average([])
        assert False, "expected a ZeroDivisionError, but none was raised"
    except ZeroDivisionError:
        pass


test_average_normal_case()
test_average_single_number()
test_average_empty_list()
print("all tests passed")

`test_average_empty_list` is the genuinely important one here — it's not testing that `average` works, it's testing that `average` *fails in a specific, expected way* for input nobody thought carefully about at first. Writing this test is what would force you to notice, and consciously decide, whether an empty list should raise an error (as it does now, by accident of how division works) or return something more deliberate, like `0` — a decision easy to skip entirely without a test explicitly asking the question.

### Failure-driven debugging

Once you have tests, they change *how* debugging works. Instead of adding scattered `print()` statements to a program and re-running the whole thing to see what looks wrong, a failing test tells you precisely which expectation broke, in isolation, without needing to run anything else — and once fixed, every other test that already passed confirms the fix didn't quietly break something else nearby.

### One problem

> Write a function `is_palindrome(text)` that checks whether a string reads the same forwards and backwards, ignoring case and spaces. Write at least four test functions for it, including at least one genuine edge case.

In [ ]:
# TODO: define is_palindrome and write test functions for it


## 62. `pytest` Fundamentals

The last chapter's tests worked, but notice they needed manual calls at the bottom (`test_add_zero()`, and so on) to actually run, and a failure just crashed the whole cell rather than reporting which specific test failed while still running the rest. `pytest` is a tool built specifically to remove both of those problems.

### Test discovery

`pytest`, run from a terminal in a project's folder, automatically finds and runs test functions on its own — no manual list of calls required — following a simple, consistent naming convention: files named `test_*.py` (or `*_test.py`), containing functions named `test_*`.

Since a notebook can't run a real terminal command interactively the way a project folder would, this chapter writes an actual test file to disk and then invokes `pytest` on it, exactly as you would from a real project.

In [ ]:
test_code = '''
def add(a, b):
    return a + b


def test_add_positive_numbers():
    assert add(2, 3) == 5


def test_add_negative_numbers():
    assert add(-2, -3) == -5


def test_add_that_will_fail():
    assert add(2, 2) == 5
'''

with open("test_math_ops.py", "w") as file:
    file.write(test_code)

In [ ]:
import subprocess

result = subprocess.run(
    ["python", "-m", "pytest", "test_math_ops.py", "-v"],
    capture_output=True, text=True
)
print(result.stdout)

Two tests pass, one fails — and notice `pytest` reports *all three* results, including exactly which one failed and why (showing the actual values `add(2, 2)` computed versus what the assertion expected), rather than stopping at the first failure the way running the functions manually in the last chapter's cell would have.

### Assertions, unchanged

`pytest` doesn't introduce a new assertion syntax — it uses plain `assert`, exactly as the last chapter did, and specifically enhances the *error message* an assertion produces on failure, showing you the actual values involved rather than just "assertion failed." This is deliberate: you don't need to learn a separate assertion API the way some other testing tools require; ordinary Python `assert` is the whole language.

### Fixtures — shared setup, without repeating it in every test

Several tests often need the same starting data. A **fixture** provides it once, and `pytest` hands it to every test function that asks for it by parameter name.

In [ ]:
test_code = '''
import pytest


@pytest.fixture
def sample_numbers():
    return [1, 2, 3, 4, 5]


def test_sum(sample_numbers):
    assert sum(sample_numbers) == 15


def test_length(sample_numbers):
    assert len(sample_numbers) == 5
'''

with open("test_with_fixture.py", "w") as file:
    file.write(test_code)

result = subprocess.run(
    ["python", "-m", "pytest", "test_with_fixture.py", "-v"],
    capture_output=True, text=True
)
print(result.stdout)

`sample_numbers`, decorated with `@pytest.fixture`, is a function `pytest` recognizes as a source of shared setup. Any test function that takes a parameter with that exact name — `test_sum(sample_numbers)` — automatically receives whatever the fixture returns, freshly, for that specific test. This removes the need to reconstruct `[1, 2, 3, 4, 5]` by hand at the top of every single test that happens to need it.

### Parameterization — running the same test against many inputs

Testing several inputs against the same logic, without copy-pasting the test function repeatedly, uses `@pytest.mark.parametrize`.

In [ ]:
test_code = '''
import pytest


def is_even(n):
    return n % 2 == 0


@pytest.mark.parametrize("number,expected", [
    (2, True),
    (3, False),
    (0, True),
    (-4, True),
    (-7, False),
])
def test_is_even(number, expected):
    assert is_even(number) == expected
'''

with open("test_parametrized.py", "w") as file:
    file.write(test_code)

result = subprocess.run(
    ["python", "-m", "pytest", "test_parametrized.py", "-v"],
    capture_output=True, text=True
)
print(result.stdout)

One test function, five separate reported results — `pytest` runs `test_is_even` once per entry in the list, substituting `number` and `expected` each time, and reports each combination as its own pass or fail. This is a direct, structured answer to the "test more than the obvious case" instinct from the last chapter — every edge case you think of becomes one more line in the list, not one more near-duplicate function.

### Interpreting a failure

Read `pytest`'s failure output for the earlier `test_add_that_will_fail` case again, carefully — it shows the actual expression that was asserted, and the actual computed values on each side of the comparison. Learning to read that output precisely, rather than skimming past it to a generic "something failed," is most of what makes tests useful for debugging in practice, exactly as the previous chapter argued.

### One problem

> Write a `pytest` test file (as a string written to disk, following this chapter's pattern) for the `is_palindrome` function from the previous chapter's problem, using `@pytest.mark.parametrize` to cover at least five cases, including at least one edge case.

In [ ]:
import subprocess

# TODO: write a parametrized pytest test file for is_palindrome, then run it


## 63. Modules and Packages

Basics Chapter 27 showed importing one `.py` file from another. As a program grows past a handful of files, a single flat folder of modules stops being enough organization, and Python provides a second, larger unit of structure: the **package**.

### Module versus package

A **module** is a single `.py` file — you've been using this term correctly since Basics. A **package** is a *folder* of modules, grouped together and importable as a single unit, typically because the modules inside it are related parts of one larger piece of functionality.

In [ ]:
import os

os.makedirs("shop/", exist_ok=True)

with open("shop/__init__.py", "w") as f:
    f.write("")

with open("shop/inventory.py", "w") as f:
    f.write(
        "def check_stock(item):\n"
        "    return f'Checking stock for {item}'\n"
    )

with open("shop/pricing.py", "w") as f:
    f.write(
        "def apply_discount(price, percent):\n"
        "    return price * (1 - percent / 100)\n"
    )

In [ ]:
from shop import inventory, pricing

print(inventory.check_stock("widget"))
print(pricing.apply_discount(100, 20))

`shop/` is a package containing two modules, `inventory.py` and `pricing.py`, importable together as `shop.inventory` and `shop.pricing` — grouped under one shared name because they're both genuinely part of "the shop" as a coherent unit, rather than unrelated files that merely happen to sit in the same folder.

### `__init__.py` — what it's actually for

The empty `__init__.py` file is what marks `shop/` as a package rather than just an ordinary folder that happens to contain Python files — its presence is what makes `from shop import inventory` work at all. It doesn't have to stay empty: code placed inside `__init__.py` runs once, the first time the package is imported, which makes it a natural place to expose a package's most commonly used pieces directly, saving callers a level of navigation.

In [ ]:
with open("shop/__init__.py", "w") as f:
    f.write(
        "from shop.inventory import check_stock\n"
        "from shop.pricing import apply_discount\n"
    )

In [ ]:
import importlib
import shop
importlib.reload(shop)

from shop import check_stock, apply_discount

print(check_stock("gadget"))
print(apply_discount(50, 10))

(`importlib.reload` is only needed here because the notebook already had `shop` imported once earlier in this session and Python normally imports a given module only once per program run — a real, fresh program starting up wouldn't need this extra step at all.) Now `check_stock` and `apply_discount` are reachable directly from `shop`, without the caller needing to know or care that they actually live in separate files underneath — exactly the kind of implementation detail Chapter 43's properties argued was worth hiding, applied here at the level of a whole package's structure instead of a single class's attributes.

### Absolute versus relative imports

The imports shown so far — `from shop import inventory`, `from shop.inventory import check_stock` — are **absolute imports**, spelled out starting from the package's own top-level name. Inside a package, a module can also import a sibling module using a **relative import**, using a leading dot to mean "relative to my own location":

```python
# inside shop/pricing.py, importing something from shop/inventory.py
from .inventory import check_stock
```

A single dot means "the current package"; two dots would mean "the parent of the current package," and so on. Relative imports are mostly a matter of convention and project style rather than one being strictly correct — many real projects prefer absolute imports throughout, specifically because they remain unambiguous and readable even if a file gets moved to a different location within the project, whereas a relative import's meaning depends on exactly where the importing file sits in the folder structure.

### Circular imports — a genuine, common problem worth recognizing

If `inventory.py` tries to import something from `pricing.py`, and `pricing.py` also tries to import something from `inventory.py`, Python can end up in a situation where neither module has finished being set up by the time the other one needs something from it — a **circular import**, and it typically surfaces as a confusing `ImportError` mentioning a name that "definitely exists," because the module defining it hadn't finished running yet when the other module tried to reach into it. The most reliable general fix is structural: it usually signals that two modules are too tightly entangled, and that some shared piece they both depend on belongs in a third module both can import from independently, rather than importing from each other directly.

### One problem

> Create a package `library/` with two modules, `books.py` (a `Book` class or function) and `search.py` (a function that takes a list of books and a search term). Set up `library/__init__.py` to expose both directly from the package, and demonstrate importing and using them.

In [ ]:
import os

# TODO: build the library package as described, then import and use it


## 64. Building a Professional Python Project

Every idea from this book so far — functions, classes, modules, packages, tests, exceptions, type hints — has appeared mostly in isolation, one notebook cell at a time. This chapter is about how they actually sit together, physically, on disk, in a real project meant to be worked on over time, potentially by more than one person.

### A sensible starting structure

```text
project/
├── package/
│   ├── __init__.py
│   ├── core.py
│   └── utils.py
├── tests/
│   ├── test_core.py
│   └── test_utils.py
├── data/
│   └── sample.csv
├── README.md
└── requirements.txt
```

Each piece here answers a specific, recurring question a project accumulates over time.

`package/` holds the actual, importable code — organized as the package Chapter 63 just introduced, so the project's own logic is cleanly separable from everything around it (tests, data, documentation) rather than mixed in with them.

`tests/`, kept as a **separate** top-level folder rather than mixed into `package/`, is a deliberate, very common convention — it means test files are never accidentally shipped or imported as part of the actual application, and `pytest` (Chapter 62) can be pointed at this one folder to run everything, without needing to distinguish real code files from test files living in the same place.

`data/` holds sample or reference data files a project might need to load — kept separate from code specifically because data files change for entirely different reasons than code does, and mixing the two makes both harder to review and version sensibly over time.

`README.md` is a plain-text (specifically, Markdown-formatted) file explaining what the project is, how to set it up, and how to run it — the very first thing anyone encountering the project, including a future version of yourself, is expected to read before touching any code.

`requirements.txt` lists the external packages the project depends on (things installed via a tool like `pip`, beyond what Python provides built-in), so someone else — or you, on a different machine — can recreate a working environment reliably, rather than guessing which packages happen to already be installed on the machine the project was originally written on.

### Why this particular separation matters

The underlying principle threading through all of this isn't specific to any one of these folders — it's the same idea Chapter 43 applied to a single class's interface, and Chapter 63 applied to a package's `__init__.py`: **group things by what they're responsible for, and keep responsibilities that change for different reasons in different places.** Test code changes when your understanding of correct behavior changes. Application code changes when behavior itself changes. Data changes when the world the program models changes. Keeping these separate means a change to one doesn't require touching, or even necessarily looking at, the others.

### A minimal worked example

Here's a tiny but genuinely complete version of this shape, built directly, to make the structure concrete rather than purely abstract.

In [ ]:
import os

os.makedirs("temperature_project/converter", exist_ok=True)
os.makedirs("temperature_project/tests", exist_ok=True)

with open("temperature_project/converter/__init__.py", "w") as f:
    f.write("from converter.core import celsius_to_fahrenheit, fahrenheit_to_celsius\n")

with open("temperature_project/converter/core.py", "w") as f:
    f.write(
        "def celsius_to_fahrenheit(celsius: float) -> float:\n"
        "    return celsius * 9 / 5 + 32\n"
        "\n"
        "def fahrenheit_to_celsius(fahrenheit: float) -> float:\n"
        "    return (fahrenheit - 32) * 5 / 9\n"
    )

with open("temperature_project/tests/test_core.py", "w") as f:
    f.write(
        "from converter.core import celsius_to_fahrenheit, fahrenheit_to_celsius\n"
        "\n"
        "def test_celsius_to_fahrenheit():\n"
        "    assert celsius_to_fahrenheit(0) == 32\n"
        "\n"
        "def test_fahrenheit_to_celsius():\n"
        "    assert fahrenheit_to_celsius(32) == 0\n"
        "\n"
        "def test_round_trip():\n"
        "    original = 37.0\n"
        "    converted = fahrenheit_to_celsius(celsius_to_fahrenheit(original))\n"
        "    assert abs(converted - original) < 0.0001\n"
    )

with open("temperature_project/README.md", "w") as f:
    f.write(
        "# Temperature Converter\n\n"
        "A small package for converting between Celsius and Fahrenheit.\n\n"
        "## Running tests\n\n"
        "From the project root: `pytest tests/`\n"
    )

print("project structure created")

In [ ]:
import subprocess

result = subprocess.run(
    ["python", "-m", "pytest", "tests/", "-v"],
    cwd="temperature_project",
    capture_output=True, text=True
)
print(result.stdout)

Every test found and reported, run from the project's own root, exactly as a real contributor to this project would run them after cloning it — this is the entire structure earning its keep: predictable, discoverable, and immediately runnable by anyone who understands the convention, without needing project-specific instructions beyond what `README.md` already states.

### What this chapter deliberately leaves out

Actually distributing this project — packaging it so it can be installed elsewhere with `pip install`, publishing it, managing version numbers and dependencies formally — is real, substantial territory of its own, and it belongs to the Advanced volume of this curriculum rather than here. What this chapter aims to leave you with is the *shape* of a well-organized project and the reasoning behind that shape, which is the part every larger, more formal packaging process ultimately builds on top of.

### One problem

> Take the `library` package from the previous chapter's problem and restructure it into a full project layout matching this chapter's pattern — `library_project/library/`, `library_project/tests/`, and a `README.md` — with at least two real `pytest` tests, and confirm they run successfully from the project root.

In [ ]:
import os

# TODO: restructure the library package into a full project layout with tests


## 65. Intermediate Capstone

Every chapter in this book taught one idea, deliberately isolated, the way every chapter in Basics did. This project asks you to do what no single chapter could: decide, on your own, which of the last thirty-five ideas actually apply to a real problem, and how they fit together.

### The task: a personal library manager

Build a small application that manages a personal book collection, persisted to disk, with a proper project structure and a real test suite.

**Requirements:**

1. **Design a class hierarchy.** At minimum, a base class representing a general library item, and at least two subclasses (for example, a physical `Book` and an `Ebook`, or a `Book` and an `Audiobook`) that share common behavior through inheritance but override at least one method meaningfully.
2. **Use a dataclass** somewhere sensible in the design — a natural candidate is a small, immutable value like a search result or a summary record, rather than the main classes with substantial behavior.
3. **Persist the collection as JSON**, loading it on startup (handling the case where the file doesn't exist yet) and saving it when the program exits or when the collection changes.
4. **Support searching** the collection by title or author, using either a comprehension or `filter`, and at least one regular expression somewhere reasonable (case-insensitive partial matching is a sensible use).
5. **Use a generator** somewhere genuinely appropriate — for instance, a function that lazily yields items matching a condition, rather than building a full list when the caller might only need the first few results.
6. **Write at least one custom exception**, raised for a genuine domain-specific error condition (attempting to add a duplicate item, searching with an empty term, or similar), and handle it explicitly somewhere.
7. **Use a context manager** for any file operations, and consider whether a small custom context manager (Chapters 50–51) would add anything over the built-in `open`.
8. **Add type hints throughout**, including at least one use of `Optional` or `Union` where genuinely appropriate.
9. **Organize the code as a real package**, following Chapter 64's structure — a package folder, a separate `tests/` folder, and a `README.md`.
10. **Write a `pytest` test suite** covering at minimum: adding an item, searching, handling the custom exception, and the JSON save/load round trip. Use `@pytest.mark.parametrize` at least once.

This is intentionally under-specified in the way a real request from someone else usually is. What exactly counts as a "physical `Book`" versus an "`Ebook`" in terms of shared and different behavior, what fields belong on the dataclass, whether the base class should be an ABC (Chapter 41) or just an ordinary base class with default behavior (Chapter 38) — these are genuine design decisions, not gaps in the instructions waiting to be filled in a single obvious way. Sketch the class hierarchy and the file layout before writing any code, the same advice the Basics capstone gave, and for the same reason: the value here is in making — and living with — your own design decisions, not in matching a hidden reference solution.

Take this one seriously. If you can build this from the ground up, on your own, using tools from across every part of this book, you're not "familiar with Intermediate Python topics" — you're someone who can actually reach for the right tool, deliberately, when a real problem calls for it. That's the entire point this whole volume has been building toward.

In [ ]:
# Your personal library manager project goes here.
# Start with a rough sketch of the package structure and class hierarchy as comments.


---

## Where this leaves you

Look back at the progression this book actually followed: comprehensions and unpacking made existing ideas more expressive; the object model gave you a way to build your own types instead of only using Python's; iterators and generators showed that a function call doesn't have to mean "compute everything, right now"; closures and decorators showed that functions are values you can build machinery out of; context managers and custom exceptions gave you honest, reliable ways to handle things going wrong; and the final stretch — real data formats, better collections, precise types, and actual tests — is what separates code that works once from code you can trust to keep working as it grows.

None of this was ever really about accumulating more syntax. It was about being able to ask, and actually answer, "what is Python doing here" — for code you write, and increasingly, for code someone else wrote that you now need to read, extend, or fix.

Advanced Python — the third volume — is where this same instinct turns toward what's happening underneath the interpreter itself: memory, performance, concurrency, and the internals this book deliberately left alone. You have exactly the foundation that volume assumes.

---

**Code With SUZH**

Understand the behavior.
Write the code.
Build with Python.

**SUZH TEAM**